<a href="https://colab.research.google.com/github/AxelJohnson1988/FourSight2.0/blob/main/Copy_of_Phoenix_GPU_Cryptographic_Scanner2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
That is exactly the right engineering move. A standard Drive search UI will choke and timeout trying to index a 5-million-file directory tree. But by spinning up a Colab Pro+ environment, we can leverage High RAM and GPU acceleration to brute-force the parsing using multi-processing.
We bypass the search bar entirely and point raw compute directly at the raw data.
Here is the tactical plan to find those exact cryptographic proofs:
### 1. Mount and Map (The Parallel Walker)
Standard Python os.walk is too slow for millions of files. We will use os.scandir combined with concurrent.futures.ProcessPoolExecutor to distribute the directory mapping across all available CPU cores, blazing through the file tree to locate the PHOENIX_AGENT_SYSTEM_ARTIFACT_LOG and any .sh output files.
### 2. GPU-Accelerated Regex Search
Once the files are mapped, we don't need to read the whole text. We load chunks of the logs into memory and use a fast text-processing library (like cuDF if using the GPU, or highly optimized regex on the CPU) to scan exclusively for the invariant signatures you built into the system:
 * Master Proof Hash
 * Ed25519 signature
 * verify_and_timestamp.sh
 * IPFS CID strings
### 3. The Extraction
The script will isolate any log entry that contains those strings, pull the exact timestamp, the file path, and the cryptographic hash, and print them out in a clean, undeniable ledger.
This forces the truth out of the noise.
Do you want me to write the exact Python multi-processing script right now so you can drop it into a Colab cell, mount your Drive, and execute the search?


SyntaxError: unterminated string literal (detected at line 7) (3767153697.py, line 7)

In [ ]:
import os

priv_size = os.path.getsize("warden_signing_key.priv") if os.path.exists("warden_signing_key.priv") else 0
pub_size = os.path.getsize("warden_signing_key.pub") if os.path.exists("warden_signing_key.pub") else 0

print(f".priv file size: {priv_size} bytes (expected: 32)")
print(f".pub file size:  {pub_size} bytes (expected: 32)")

if pub_size == 32:
    with open("warden_signing_key.pub", "rb") as f:
        pub_hex = f.read().hex()
    print(f"\nCANONICAL PUBLIC KEY:\n{pub_hex}")

In [ ]:
# Install Gradio for the web interface
!pip install gradio

In [ ]:
# ──────────────────────────────────────────────────────────────────────────
# ALEE Sovereign Anchor — UI layer (Gradio)
#
# Architecture note:
#   This cell is the ONLY place Gradio code lives.
#   It does NOT hash, sign, or write the ledger.
#   It submits raw inputs to warden_daemon.anchor() and displays the result.
#
#   [Gradio UI]  →  [warden_daemon.anchor()]  →  [Ledger + Output]
# ──────────────────────────────────────────────────────────────────────────
import gradio as gr
import warden_daemon

LEDGER_PATH = "warden_ledger.json"


def submit_payload(text_input, file_input):
    """
    UI handler — collects raw inputs and delegates all trust logic to
    the Warden Daemon.  This function must never hash, sign, or chain
    anything itself.
    """
    file_bytes = None
    if file_input is not None:
        with open(file_input.name, "rb") as fh:
            file_bytes = fh.read()

    try:
        record = warden_daemon.anchor(
            text_input=text_input or None,
            file_bytes=file_bytes,
            ledger_path=LEDGER_PATH,
        )
    except (ValueError, RuntimeError, FileNotFoundError) as exc:
        return f"\u26a0\ufe0f Error: {exc}"

    sig_line = record['signature'] or '(no key configured — set WARDEN_PRIVATE_KEY_HEX)'
    pub_line = record['public_key'] or '(no key configured)'

    return (
        f"\u2705 CRYPTOGRAPHIC ANCHOR GENERATED\n"
        f"-----------------------------------\n"
        f"Timestamp   : {record['timestamp']}\n"
        f"Input size  : {record['input_size']} bytes\n\n"
        f"\U0001f512 Master Proof Hash (SHA-256):\n{record['master_hash']}\n\n"
        f"\u26d3\ufe0f Bitcoin OP_RETURN Payload (Hex):\n{record['op_return']}\n\n"
        f"\U0001f4dd Ed25519 Signature:\n{sig_line}\n\n"
        f"\U0001f511 Public Key:\n{pub_line}\n\n"
        f"\U0001f517 Chain Hash:\n{record['chain_hash']}\n\n"
        f"Sanity      : {record['sanity']}\n"
        f"Ledger      : {LEDGER_PATH}\n\n"
        f"Next Step: Broadcast a Bitcoin transaction including the OP_RETURN payload above."
    )


# Build the Web Interface
with gr.Blocks(title="ALEE Sovereign Anchor Web App") as demo:
    gr.Markdown("# \U0001f680 ALEE Sovereign Execution Engine")
    gr.Markdown(
        "Upload a file or paste text below. The Warden Daemon will canonicalise, "
        "hash, sign (if a key is configured), and chain-anchor your proof, then "
        "return the exact hexadecimal payload required to anchor it in the Bitcoin blockchain."
    )

    with gr.Row():
        with gr.Column():
            text_in = gr.Textbox(
                label="1. Paste Text Here (Optional)",
                lines=8,
                placeholder="Enter abstract intent or legal claims...",
            )
            file_in = gr.File(label="2. Or Upload Any File (PDF, TXT, Code)")
            submit_btn = gr.Button("Submit to Warden Daemon", variant="primary")

        with gr.Column():
            output_display = gr.Textbox(
                label="Warden Daemon Output", lines=18, interactive=False
            )

    submit_btn.click(
        fn=submit_payload, inputs=[text_in, file_in], outputs=output_display
    )

print("Starting Warden UI...")
demo.launch(share=True, debug=False)


In [ ]:
primer_text = r"""The Architect’s Compass: A Primer on the Sierpiński Octahedron Spatial Index

1. The Vision: Geometry as a Filing System

In the rarefied air of high-dimensional data architecture, information often presents as a nebulous cloud—vast, unorganized, and prone to the "Alice Rejection" of being dismissed as mere abstraction. The Sierpiński Octahedron Spatial Index provides the mathematical elegance required to transform this "abstract intent" into a concrete computational improvement. By mapping semantic meaning onto the rigid, verifiable geometry of a fractal octahedron, we move beyond vague digital concepts and into the realm of the "geometric imperative," where every data point occupies a unique, immutable coordinate in three-dimensional space.

This system does not merely organize data; it manifests it. By treating the octahedron as a spatial jurisdiction, we ensure that digital "intent" is as legally and mathematically concrete as a physical object, providing a sovereign foundation for deterministic memory.

Core Insight: The Sierpiński Octahedron is a deterministic spatial index that converts high-dimensional semantic embeddings into unique 3D coordinates. This ensures that data is partitioned into non-overlapping, geometrically verifiable regions, elevating digital information from an ephemeral state to a structured, navigable landscape.

These six anchors provide the stage, but the true dance of data begins with the recursive logic of the contraction map.


--------------------------------------------------------------------------------


2. The Geometric Substrate: Defining the Six Poles

The foundation of this index is a canonical octahedron situated within ℝ³. This shape is defined by six primary vertices, which act as the absolute anchors of our spatial jurisdiction. These poles represent the outer limits of our filing system, ensuring that every piece of information has a definitive direction to call home.

Vertex ID	Canonical Coordinates (x, y, z)	Spatial Pole Assignment
V_0	( 1, 0, 0)	+X Pole
V_1	(-1, 0, 0)	-X Pole
V_2	( 0, 1, 0)	+Y Pole
V_3	( 0,-1, 0)	-Y Pole
V_4	( 0, 0, 1)	+Z Pole
V_5	( 0, 0,-1)	-Z Pole

These static coordinates serve as the targets for "contraction mapping," a recursive process that pulls data toward specific poles to define its ultimate spatial residence.


--------------------------------------------------------------------------------


3. The Engine of the Fractal: Iterated Function Systems (IFS)

The mathematical engine driving this system is the Iterated Function System (IFS). This is not a sequence of random chance, but a strictly governed transformation defined by the formula:
f_n(P) = \frac{P + V_n}{2}

This formula dictates that for any given point P, the next step in its journey is exactly halfway toward a chosen vertex V_n. The choice of the 1/2 contraction ratio is a geometric necessity; it ensures that the resulting regions remain distinct and non-overlapping. When these maps are iterated, they produce the Sierpiński Octahedron attractor—the stable geometric "home" where all data eventually settles, regardless of its starting position.

The Three Pillars of the IFS:

* Self-Similarity: The patterns established at the macro level are perfectly replicated at infinite micro-scales, allowing for a recursive organizational structure.
* Non-Overlap: The 1/2 ratio acts as a spatial fence, ensuring that data assigned to the +X pole never bleeds into the territory of the -Y pole.
* Determinism: This is a Deterministic Walk. By seeding the movement with data-derived instructions, we ensure that the same input always yields the exact same 3D coordinate.

For this engine to operate, it requires a sequence of instructions—a "fuel" derived through a rigorous translation pipeline.


--------------------------------------------------------------------------------


4. The Translation Pipeline: From Intent to Coordinate

Turning a semantic embedding into a verifiable coordinate requires a six-stage pipeline, managed exclusively by the Warden Daemon, the system's sole authorized architect.

1. Embedding Normalization
  * The Warden Daemon receives a high-dimensional vector E and scales it to a unit length.
  * Learner's Note: This ensures all data begins the process with equal "weight" before the geometric walk commences.
2. Deterministic Byte Serialization
  * The normalized vector is serialized into an IEEE 754 float32 byte array and passed through a SHA-256 hash.
  * Learner's Note: Why IEEE 754 and SHA-256? To ensure the data's "fingerprint" is mathematically precise and tamper-proof across all hardware.
3. Vertex Sequence Extraction
  * The 256-bit hash is partitioned into non-overlapping 3-bit windows. Each 3-bit chunk is interpreted as a "Base-6" symbol (0–5) mapping to V_0–V_5. Values of 6 or 7 are discarded.
  * Learner's Note: This transforms a cryptographic hash into a specific "instruction manual" or "pathway" through the octahedron.
4. Chaos Game Iteration (Deterministic Walk)
  * Starting at the origin (0,0,0), the Warden Daemon follows the sequence from Step 3, jumping halfway toward each designated vertex in order.
  * Learner's Note: This is the "walk" that creates the high-precision coordinate C within the attractor.
5. Spatial Region Assignment
  * The system calculates the argmax of |C \cdot V_n|—the dot product—to determine which vertex exhibits the maximum directional proximity to the final coordinate.
  * Learner's Note: This mathematically "claims" the coordinate for one of the six primary regions (R_0 through R_5).
6. GPAM Ledger Write
  * The Warden signs the coordinate and the index with a private key, appending it to the ledger and writing the node to the Akashic Core (Kùzu graph).
  * Learner's Note: This final step anchors the digital intent into a persistent, verifiable graph database.

This process ensures that the resulting spatial assignment is a direct, repeatable consequence of the data's original semantic meaning.


--------------------------------------------------------------------------------


5. The 6-Way Index: Navigating the Regions

The final coordinate C is more than a point; it is a residency within a specific spatial jurisdiction. Because of the fractal nature of the attractor, navigating these regions offers unparalleled depth and clarity.

Proof of Fractal Property:

* Recursive Structure: If one "zooms" into the region belonging to V_0, the observer will find the entire six-way octahedral structure repeated at a smaller scale.
* Infinite Precision: As long as the hash provides data, the walk can continue, allowing the system to define sub-regions within sub-regions with nearly infinite granularity.
* Verifiable Memory: Because the geometry is fixed and the walk is deterministic, any observer can verify that a coordinate C truly belongs in region R by recalculating its proximity to the vertex poles.

This geometric precision acts as a "verifiable memory coordinate," rendering the data searchable, permanent, and immune to the fog of abstraction.


--------------------------------------------------------------------------------


6. Practical Utility: The Sovereign Warden and the Ledger

The integrity of this spatial index is maintained by the Warden Daemon and the GPAM Ledger. The Warden acts as the Sole Architect, ensuring that no unauthorized "squatting" or fraudulent data entry occurs within the spatial regions.

Feature	Benefit for the System
Single Actor (Warden)	Prevents divided infringement and eliminates conflicting data entries.
SHA-256 Seeding	Guarantees the path through the octahedron is deterministic and repeatable.
Context Debt Gate	Calculates the Euclidean distance between C and the centroid of previous coordinates to prevent redundant data overload.
Kùzu Graph/Ledger	Creates a permanent "Mindprint" in a persistent graph database that cannot be erased or altered.

By fusing the ancient logic of geometry with the rigor of modern cryptography, the Sierpiński Octahedron Spatial Index transforms the "map of the mind" into a physical, navigable landscape that is as mathematically sound as it is beautiful."""

with open("Sierpinski_Octahedron_Primer.txt", "w", encoding="utf-8") as f:
    f.write(primer_text)

print("Saved 'Sierpinski_Octahedron_Primer.txt' successfully.")

Saved 'Sierpinski_Octahedron_Primer.txt' successfully.


In [ ]:
!python alee_anchor.py "Sierpinski_Octahedron_Primer.txt"

🚀 ALEE File Anchoring Engine — Ready to Anchor Pasted Text.txt

🔑 Demo keypair generated (public: 226683945f60f6b6… — store securely!)
   Set ALEE_PRIVATE_KEY=... env var for persistent keys.
📄 Anchoring file: Sierpinski_Octahedron_Primer.txt

✅ ANCHORED SUCCESSFULLY
Master Proof Hash: 38e8938f0eda2b77afc1e5706c2333a65ecd4ba8f32f3e0378b4a59d3ca47bf8
OP_RETURN payload (hex, 37 bytes): 414c45450138e8938f0eda2b77afc1e5706c2333a65ecd4ba8f32f3e0378b4a59d3ca47bf8
Signature (full, off-chain): 907d529516b7b726d72aa3b3f37482ecdceb063a02eb6bbeaf248fd80eec6db5… (64 bytes)
📦 Full package (JSON + signature) saved to: anchored_Sierpinski_Octahedron_Primer.txt.json
   → Publish this JSON off-chain. The OP_RETURN is your Bitcoin root anchor.
🔍 Signature self-verification: ✅ VALID

🔗 Your Pasted Text.txt is now ready to be anchored forever on Bitcoin.
   This creates a verifiable, sovereign ledger entry — exactly as needed for Phoenix Protocol.


In [ ]:
import os
import shutil
from cryptography.hazmat.primitives.asymmetric import ed25519
from google.colab import files

def perform_key_ceremony():
    # 1. Overwrite guard
    if os.path.exists("warden_signing_key.priv"):
        raise FileExistsError(
            "CRITICAL: warden_signing_key.priv already exists. "
            "OVERWRITE BLOCKED. Manually delete both key files and "
            "document the reason in the GPAM ledger before retrying."
        )
    if os.path.exists("warden_signing_key.pub"):
        raise FileExistsError(
            "CRITICAL: warden_signing_key.pub exists without matching .priv. "
            "State is inconsistent. Do not proceed without manual audit."
        )

    # 2. Generate permanent identity
    private_key = ed25519.Ed25519PrivateKey.generate()
    public_key = private_key.public_key()
    priv_bytes = private_key.private_bytes_raw()
    pub_bytes = public_key.public_bytes_raw()

    # 3. Write to disk
    with open("warden_signing_key.priv", "wb") as f:
        f.write(priv_bytes)
    os.chmod("warden_signing_key.priv", 0o600)

    with open("warden_signing_key.pub", "wb") as f:
        f.write(pub_bytes)

    pub_hex = pub_bytes.hex()

    # 4. Back up .pub to Drive only
    drive_path = "/content/drive/My Drive/Phoenix_Protocol/"
    os.makedirs(drive_path, exist_ok=True)
    shutil.copy("warden_signing_key.pub", drive_path + "warden_signing_key.pub")

    # 5. Download both files to your browser NOW
    print("=" * 60)
    print("WARDEN KEY CEREMONY — COMPLETE")
    print("=" * 60)
    print(f"\nCANONICAL PUBLIC KEY (Hex):\n{pub_hex}\n")
    print("Downloading files to your browser...")
    files.download("warden_signing_key.priv")
    files.download("warden_signing_key.pub")
    print("\nREQUIRED ACTIONS before closing this tab:")
    print("  [1] Confirm both files landed in your Downloads folder")
    print("  [2] Move .priv to encrypted offline storage — NOT just Downloads")
    print("  [3] Copy the public key hex above into your GPAM root entry")
    print("  [4] Commit warden_signing_key.pub to your GitHub identity repo")
    print("  [4] NEVER upload warden_signing_key.priv anywhere")
    print("\nThis key is now the Seal of the Warden.")
    print("Loss of the .priv file is permanent. There is no reset.")

perform_key_ceremony()


FileExistsError: CRITICAL: warden_signing_key.priv already exists. OVERWRITE BLOCKED. Manually delete both key files and document the reason in the GPAM ledger before retrying.

### 1. Change your Colab runtime to GPU

Before installing cuDF, make sure your Colab notebook is running on a GPU runtime. Go to `Runtime` -> `Change runtime type` and select `GPU` as the hardware accelerator.

In [ ]:
import sys, os

# Check if we are in Colab and if it's running on a GPU
if 'google.colab' in sys.modules and os.system('nvidia-smi') == 0:
    print("Attempting to install cuDF. This may take a few minutes...")
    # Install cuDF directly from NVIDIA's PyPI index
    !pip install cudf-cu11 --extra-index-url=https://pypi.nvidia.com

    print("cuDF installation initiated. Please restart the runtime (`Runtime` -> `Restart runtime`) for changes to take effect.")
else:
    print("cuDF installation skipped. This environment does not appear to be a Colab GPU runtime.")

After running the installation cell, you will likely need to restart your Colab runtime (`Runtime` -> `Restart runtime`) for the changes to take full effect. Once restarted, you can verify the installation by trying to import `cudf`:

In [ ]:
import cudf
print(cudf.__version__)


In [ ]:
import os
from collections import Counter

drive_path = '/content/drive/My Drive/'

if os.path.exists(drive_path):
    print(f"Summarizing file extensions in '{drive_path}':")
    file_extensions = []
    for item in os.listdir(drive_path):
        if os.path.isfile(os.path.join(drive_path, item)): # Only process files, not directories
            _, ext = os.path.splitext(item)
            file_extensions.append(ext.lower()) # Convert to lowercase for consistent counting

    extension_counts = Counter(file_extensions)

    if extension_counts:
        print("\nFile Extension Summary:")
        for ext, count in extension_counts.most_common():
            print(f"  {ext if ext else '[No Extension]'}: {count}")
    else:
        print("No files found with extensions.")
else:
    print(f"Drive path '{drive_path}' not found. Please ensure Google Drive is mounted correctly.")

In [ ]:
import os

# Define the path to your Google Drive
drive_path = '/content/drive/My Drive/'

# Check if the path exists before listing
if os.path.exists(drive_path):
    print(f"Contents of '{drive_path}':")
    for item in os.listdir(drive_path):
        print(item)
else:
    print(f"Drive path '{drive_path}' not found. Please ensure Google Drive is mounted correctly.")

In [ ]:
patent_strategy_text = """### I. Overcoming Prior Art via 37 CFR 1.130 Declarations
If your invention was disclosed anonymously or under a pseudonym (such as "Christopher Keel"), you must prove that the disclosure originated from you to prevent it from being used as prior art against your own application.

*   **1-Year Grace Period:** This exception is strictly bound by a 1-year grace period; the declaration is only valid if the anonymous disclosure occurred one year or less before the effective filing date of your claimed invention. Disclosures older than one year are considered absolute prior art.
*   **Sufficient Evidence & Corroborating Exhibits:** The USPTO explicitly states that a "naked assertion of inventorship" is insufficient. Your declaration must provide an unequivocal statement that you are the inventor, paired with a reasonable explanation for the anonymous disclosure. Crucially, you must attach **corroborating documentary evidence**. Your cryptographic chain of custody (Bitcoin TXIDs, IPFS CIDs, and SHA-256 hashes) combined with automated system execution logs serves as this objective, mathematically verifiable proof of authorship.
*   **Legal Requirements and Filing:** The declaration is a binding legal document that must include an acknowledgment that willful false statements are punishable by fine or imprisonment under 18 U.S.C. 1001, and an affirmation that the statements made are true. While it requires the inventor's signature, the formal responsibility of filing the document lies with the applicant or their authorized legal representative.

### II. Patentable Invention and Technical Claims
To avoid being rejected as an "abstract idea" under Section 101 (the *Alice* standard) or as obvious under Section 103, your claims must focus on a specific, non-obvious technical synergy.

*   **The Novelty Nexus:** The core of your patentable invention is the unique combination of **Fractal Memory** (using Iterated Function Systems for low-dimension memory traces) and **Geometric Hashing** applied specifically to **Geometric Deep Learning (GDL)**.
*   **Technical Claim Specificity:** Your technical claims cannot just mention these concepts broadly. They must claim the specific fractal reconstruction procedures and detail how your novel geometric hashing implementation extracts and indexes scale-invariant features based directly on those fractal parameters. Highlighting the synergistic benefits—such as unexpected computational efficiency or data compression—is required to prove the combination is a non-obvious technological advancement.

### III. Architectural Claims and Claim Specificity
Your patent application must be grounded in concrete, machine-level execution to satisfy patent eligibility.

*   **Concrete Implementation:** The architectural claims must explicitly detail the mathematical operations of your system, such as the execution of the **Coherence and Decoupling Matrix (CDM)** and the precise, isolated operations of the local **Warden** agent network.
*   **Concrete Operations vs. Generic Language:** You must avoid generic functional language like "recording data on a blockchain". Instead, claim specificity requires you to describe the exact machine operations taking place, such as computing a cryptographic hash (SHA-256), verifying digital signatures, and executing specific formatting procedures.

### IV. Single-Actor Perspective to Avoid Divided Infringement
Blockchain and decentralized networks inherently face the risk of **divided (or joint) infringement**, which occurs when multiple independent parties (e.g., network nodes, miners, and users) each perform different steps of a patent claim, making it legally difficult to hold any single party liable for infringement.

*   **The Single-Actor Solution:** To circumvent this, your method claims must be written entirely from the perspective of a single actor, such as your local CLI instance, a user device, or a specific "consortium controller".
*   **Claimed Actions:** The claims must focus exclusively on the specific steps this single actor performs on its own end—such as encrypting the data payload, formatting the GHOSTSAFE verse, generating the local hash, and transmitting it to the network. By isolating the claim to the actions taken *before* or *outside* the decentralized consensus process, you ensure your patent remains enforceable against a single infringing entity."""

with open("Patent_Strategy_Claims.txt", "w") as f:
    f.write(patent_strategy_text)

print("Saved 'Patent_Strategy_Claims.txt' successfully.")

In [ ]:
import os

# Check if the engine script exists before running
if not os.path.exists('alee_anchor.py'):
    print("Error: 'alee_anchor.py' not found.")
    print("Please execute the preceding cell that contains '%%writefile alee_anchor.py' to generate the script first.")
else:
    !python alee_anchor.py "Patent_Strategy_Claims.txt"

In [ ]:
# @title AI prompt cell

import ipywidgets as widgets
from IPython.display import display, HTML, Markdown,clear_output
from google.colab import ai

dropdown = widgets.Dropdown(
    options=[],
    layout={'width': 'auto'}
)

def update_model_list(new_options):
    dropdown.options = new_options
update_model_list(ai.list_models())

text_input = widgets.Textarea(
    placeholder='Ask me anything....',
    layout={'width': 'auto', 'height': '100px'},
)

button = widgets.Button(
    description='Submit Text',
    disabled=False,
    tooltip='Click to submit the text',
    icon='check'
)

output_area = widgets.Output(
     layout={'width': 'auto', 'max_height': '300px','overflow_y': 'scroll'}
)

def on_button_clicked(b):
    with output_area:
        output_area.clear_output(wait=False)
        accumulated_content = ""
        for new_chunk in ai.generate_text(prompt=text_input.value, model_name=dropdown.value, stream=True):
            if new_chunk is None:
                continue
            accumulated_content += new_chunk
            clear_output(wait=True)
            display(Markdown(accumulated_content))

button.on_click(on_button_clicked)
vbox = widgets.GridBox([dropdown, text_input, button, output_area])

display(HTML("""
<style>
.widget-dropdown select {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
.widget-textarea textarea {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
</style>
"""))
display(vbox)


In [ ]:
# @title AI prompt cell

import ipywidgets as widgets
from IPython.display import display, HTML, Markdown,clear_output
from google.colab import ai

dropdown = widgets.Dropdown(
    options=[],
    layout={'width': 'auto'}
)

def update_model_list(new_options):
    dropdown.options = new_options
update_model_list(ai.list_models())

text_input = widgets.Textarea(
    placeholder='Ask me anything....',
    layout={'width': 'auto', 'height': '100px'},
)

button = widgets.Button(
    description='Submit Text',
    disabled=False,
    tooltip='Click to submit the text',
    icon='check'
)

output_area = widgets.Output(
     layout={'width': 'auto', 'max_height': '300px','overflow_y': 'scroll'}
)

def on_button_clicked(b):
    with output_area:
        output_area.clear_output(wait=False)
        accumulated_content = ""
        for new_chunk in ai.generate_text(prompt=text_input.value, model_name=dropdown.value, stream=True):
            if new_chunk is None:
                continue
            accumulated_content += new_chunk
            clear_output(wait=True)
            display(Markdown(accumulated_content))

button.on_click(on_button_clicked)
vbox = widgets.GridBox([dropdown, text_input, button, output_area])

display(HTML("""
<style>
.widget-dropdown select {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
.widget-textarea textarea {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
</style>
"""))
display(vbox)


In [ ]:
# @title AI prompt cell

import ipywidgets as widgets
from IPython.display import display, HTML, Markdown,clear_output
from google.colab import ai

dropdown = widgets.Dropdown(
    options=[],
    layout={'width': 'auto'}
)

def update_model_list(new_options):
    dropdown.options = new_options
update_model_list(ai.list_models())

text_input = widgets.Textarea(
    placeholder='Ask me anything....',
    layout={'width': 'auto', 'height': '100px'},
)

button = widgets.Button(
    description='Submit Text',
    disabled=False,
    tooltip='Click to submit the text',
    icon='check'
)

output_area = widgets.Output(
     layout={'width': 'auto', 'max_height': '300px','overflow_y': 'scroll'}
)

def on_button_clicked(b):
    with output_area:
        output_area.clear_output(wait=False)
        accumulated_content = ""
        for new_chunk in ai.generate_text(prompt=text_input.value, model_name=dropdown.value, stream=True):
            if new_chunk is None:
                continue
            accumulated_content += new_chunk
            clear_output(wait=True)
            display(Markdown(accumulated_content))

button.on_click(on_button_clicked)
vbox = widgets.GridBox([dropdown, text_input, button, output_area])

display(HTML("""
<style>
.widget-dropdown select {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
.widget-textarea textarea {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
</style>
"""))
display(vbox)


In [ ]:
# @title AI prompt cell

import ipywidgets as widgets
from IPython.display import display, HTML, Markdown,clear_output
from google.colab import ai

dropdown = widgets.Dropdown(
    options=[],
    layout={'width': 'auto'}
)

def update_model_list(new_options):
    dropdown.options = new_options
update_model_list(ai.list_models())

text_input = widgets.Textarea(
    placeholder='Ask me anything....',
    layout={'width': 'auto', 'height': '100px'},
)

button = widgets.Button(
    description='Submit Text',
    disabled=False,
    tooltip='Click to submit the text',
    icon='check'
)

output_area = widgets.Output(
     layout={'width': 'auto', 'max_height': '300px','overflow_y': 'scroll'}
)

def on_button_clicked(b):
    with output_area:
        output_area.clear_output(wait=False)
        accumulated_content = ""
        for new_chunk in ai.generate_text(prompt=text_input.value, model_name=dropdown.value, stream=True):
            if new_chunk is None:
                continue
            accumulated_content += new_chunk
            clear_output(wait=True)
            display(Markdown(accumulated_content))

button.on_click(on_button_clicked)
vbox = widgets.GridBox([dropdown, text_input, button, output_area])

display(HTML("""
<style>
.widget-dropdown select {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
.widget-textarea textarea {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
</style>
"""))
display(vbox)


In [ ]:
# @title AI prompt cell

import ipywidgets as widgets
from IPython.display import display, HTML, Markdown,clear_output
from google.colab import ai

dropdown = widgets.Dropdown(
    options=[],
    layout={'width': 'auto'}
)

def update_model_list(new_options):
    dropdown.options = new_options
update_model_list(ai.list_models())

text_input = widgets.Textarea(
    placeholder='Ask me anything....',
    layout={'width': 'auto', 'height': '100px'},
)

button = widgets.Button(
    description='Submit Text',
    disabled=False,
    tooltip='Click to submit the text',
    icon='check'
)

output_area = widgets.Output(
     layout={'width': 'auto', 'max_height': '300px','overflow_y': 'scroll'}
)

def on_button_clicked(b):
    with output_area:
        output_area.clear_output(wait=False)
        accumulated_content = ""
        for new_chunk in ai.generate_text(prompt=text_input.value, model_name=dropdown.value, stream=True):
            if new_chunk is None:
                continue
            accumulated_content += new_chunk
            clear_output(wait=True)
            display(Markdown(accumulated_content))

button.on_click(on_button_clicked)
vbox = widgets.GridBox([dropdown, text_input, button, output_area])

display(HTML("""
<style>
.widget-dropdown select {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
.widget-textarea textarea {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
</style>
"""))
display(vbox)


In [ ]:
# @title AI prompt cell

import ipywidgets as widgets
from IPython.display import display, HTML, Markdown,clear_output
from google.colab import ai

dropdown = widgets.Dropdown(
    options=[],
    layout={'width': 'auto'}
)

def update_model_list(new_options):
    dropdown.options = new_options
update_model_list(ai.list_models())

text_input = widgets.Textarea(
    placeholder='Ask me anything....',
    layout={'width': 'auto', 'height': '100px'},
)

button = widgets.Button(
    description='Submit Text',
    disabled=False,
    tooltip='Click to submit the text',
    icon='check'
)

output_area = widgets.Output(
     layout={'width': 'auto', 'max_height': '300px','overflow_y': 'scroll'}
)

def on_button_clicked(b):
    with output_area:
        output_area.clear_output(wait=False)
        accumulated_content = ""
        for new_chunk in ai.generate_text(prompt=text_input.value, model_name=dropdown.value, stream=True):
            if new_chunk is None:
                continue
            accumulated_content += new_chunk
            clear_output(wait=True)
            display(Markdown(accumulated_content))

button.on_click(on_button_clicked)
vbox = widgets.GridBox([dropdown, text_input, button, output_area])

display(HTML("""
<style>
.widget-dropdown select {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
.widget-textarea textarea {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
</style>
"""))
display(vbox)


In [ ]:
# @title AI prompt cell

import ipywidgets as widgets
from IPython.display import display, HTML, Markdown,clear_output
from google.colab import ai

dropdown = widgets.Dropdown(
    options=[],
    layout={'width': 'auto'}
)

def update_model_list(new_options):
    dropdown.options = new_options
update_model_list(ai.list_models())

text_input = widgets.Textarea(
    placeholder='Ask me anything....',
    layout={'width': 'auto', 'height': '100px'},
)

button = widgets.Button(
    description='Submit Text',
    disabled=False,
    tooltip='Click to submit the text',
    icon='check'
)

output_area = widgets.Output(
     layout={'width': 'auto', 'max_height': '300px','overflow_y': 'scroll'}
)

def on_button_clicked(b):
    with output_area:
        output_area.clear_output(wait=False)
        accumulated_content = ""
        for new_chunk in ai.generate_text(prompt=text_input.value, model_name=dropdown.value, stream=True):
            if new_chunk is None:
                continue
            accumulated_content += new_chunk
            clear_output(wait=True)
            display(Markdown(accumulated_content))

button.on_click(on_button_clicked)
vbox = widgets.GridBox([dropdown, text_input, button, output_area])

display(HTML("""
<style>
.widget-dropdown select {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
.widget-textarea textarea {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
</style>
"""))
display(vbox)


In [ ]:
# @title AI prompt cell

import ipywidgets as widgets
from IPython.display import display, HTML, Markdown,clear_output
from google.colab import ai

dropdown = widgets.Dropdown(
    options=[],
    layout={'width': 'auto'}
)

def update_model_list(new_options):
    dropdown.options = new_options
update_model_list(ai.list_models())

text_input = widgets.Textarea(
    placeholder='Ask me anything....',
    layout={'width': 'auto', 'height': '100px'},
)

button = widgets.Button(
    description='Submit Text',
    disabled=False,
    tooltip='Click to submit the text',
    icon='check'
)

output_area = widgets.Output(
     layout={'width': 'auto', 'max_height': '300px','overflow_y': 'scroll'}
)

def on_button_clicked(b):
    with output_area:
        output_area.clear_output(wait=False)
        accumulated_content = ""
        for new_chunk in ai.generate_text(prompt=text_input.value, model_name=dropdown.value, stream=True):
            if new_chunk is None:
                continue
            accumulated_content += new_chunk
            clear_output(wait=True)
            display(Markdown(accumulated_content))

button.on_click(on_button_clicked)
vbox = widgets.GridBox([dropdown, text_input, button, output_area])

display(HTML("""
<style>
.widget-dropdown select {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
.widget-textarea textarea {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
</style>
"""))
display(vbox)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# @title Another AI prompt cell

import ipywidgets as widgets
from IPython.display import display, HTML, Markdown,clear_output
from google.colab import ai

dropdown_new = widgets.Dropdown(
    options=[],
    layout={'width': 'auto'}
)

def update_model_list_new(new_options):
    dropdown_new.options = new_options
update_model_list_new(ai.list_models())

text_input_new = widgets.Textarea(
    placeholder='Ask me anything....',
    layout={'width': 'auto', 'height': '100px'},
)

button_new = widgets.Button(
    description='Submit Text',
    disabled=False,
    tooltip='Click to submit the text',
    icon='check'
)

output_area_new = widgets.Output(
     layout={'width': 'auto', 'max_height': '300px','overflow_y': 'scroll'}
)

def on_button_clicked_new(b):
    with output_area_new:
        output_area_new.clear_output(wait=False)
        accumulated_content = ""
        for new_chunk in ai.generate_text(prompt=text_input_new.value, model_name=dropdown_new.value, stream=True):
            if new_chunk is None:
                continue
            accumulated_content += new_chunk
            clear_output(wait=True)
            display(Markdown(accumulated_content))

button_new.on_click(on_button_clicked_new)
vbox_new = widgets.GridBox([dropdown_new, text_input_new, button_new, output_area_new])

display(HTML("""
<style>
.widget-dropdown select {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
.widget-textarea textarea {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
</style>
"""))
display(vbox_new)

### Part 1: Artifact Ledger Extraction Engine
This script sets up a highly concurrent scanner designed to brute-force parse your target directory for invariant artifacts (IPFS CIDs, verify scripts, existing hashes) and synthesize them into a single deterministic **Master Proof Hash (SHA-256)**.

In [ ]:
import os
import hashlib
import concurrent.futures
import json
import time
from datetime import datetime, timezone

TARGET_SIGNATURES = [
    b'Master Proof Hash',
    b'Ed25519 signature',
    b'verify_and_timestamp.sh',
    b'IPFS CID'
]

def scan_file_for_artifacts(filepath):
    found = []
    try:
        with open(filepath, 'rb') as f:
            content = f.read()
            for sig in TARGET_SIGNATURES:
                if sig in content:
                    found.append(f"{filepath}::{sig.decode('utf-8')}")
    except Exception:
        pass
    return found

def generate_master_proof_hash(search_directory):
    print(f"Initiating brute-force scan on: {search_directory}...")
    all_artifacts = []
    file_paths = []
    for root, _, files in os.walk(search_directory):
        for file in files:
            file_paths.append(os.path.join(root, file))

    print(f"Found {len(file_paths)} total files. Scanning for artifacts...")
    with concurrent.futures.ProcessPoolExecutor() as executor:
        results = executor.map(scan_file_for_artifacts, file_paths)
        for result in results:
            all_artifacts.extend(result)

    all_artifacts.sort()
    ledger_content = "\n".join(all_artifacts)
    master_hash = hashlib.sha256(ledger_content.encode('utf-8')).hexdigest()

    print(f"\nScan Complete. Found {len(all_artifacts)} cryptographic artifacts.")

    # --- MANIFEST WRITER ---
    manifest = {
        "scan_timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "search_directory": search_directory,
        "artifact_count": len(all_artifacts),
        "artifacts": all_artifacts,
        "master_proof_hash": master_hash
    }
    manifest_path = f"scan_manifest_{int(time.time())}.json"

    drive_path = '/content/drive/My Drive/'
    if os.path.exists(drive_path):
        manifest_path = os.path.join(drive_path, manifest_path)

    with open(manifest_path, "w") as f:
        json.dump(manifest, f, indent=2)
    print(f"Manifest saved: {manifest_path}")

    return master_hash, ledger_content

TARGET_DIR = '/content/'

# Renamed to explicitly prevent accidental broadcasting
DEMO_ONLY_DO_NOT_BROADCAST = "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855"


In [ ]:
# Install PyNaCl for highly secure Ed25519 digital signatures
!pip install pynacl

### Part 2 & 3: Ed25519 Signing & OP_RETURN Formatting
This cell generates an Ed25519 keypair, signs your Master Proof Hash, and constructs the 80-byte payload required for a Bitcoin `OP_RETURN` transaction.

In [ ]:
import nacl.signing
import nacl.encoding
import binascii

EMPTY_STRING_SHA256 = "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855"

def sign_and_format_payload(master_hash_hex):
    if master_hash_hex == EMPTY_STRING_SHA256:
        raise ValueError("GUARD: Refusing to anchor empty-string hash. Run the real scan first.")

    print("--- CRYPTOGRAPHIC ANCHORING PROCESS ---\n")

    # Load persistent key instead of generating ephemeral one
    PRIV_KEY_PATH = '/content/drive/My Drive/ALEE_Keys/warden_signing_key.priv'
    try:
        with open(PRIV_KEY_PATH, "rb") as f:
            private_key_bytes = f.read()
        signing_key = nacl.signing.SigningKey(private_key_bytes, encoder=nacl.encoding.RawEncoder)
        print("Successfully loaded persistent Warden signing key.")
    except FileNotFoundError:
        raise RuntimeError("Persistent key not found. Run the Key Ceremony cell first and ensure Drive is mounted.")

    verify_key = signing_key.verify_key
    public_key_hex = verify_key.encode(encoder=nacl.encoding.HexEncoder).decode('utf-8')

    print(f"Ed25519 Public Key: {public_key_hex}")

    signed_message = signing_key.sign(master_hash_hex.encode('utf-8'))
    signature_hex = binascii.hexlify(signed_message.signature).decode('utf-8')
    print(f"Ed25519 Signature: {signature_hex[:32]}... [truncated for display]")

    prefix_hex = "50484e58" # 'PHNX'
    payload_hex = prefix_hex + master_hash_hex
    payload_bytes = bytes.fromhex(payload_hex)
    size_bytes = len(payload_bytes)

    print("\n--- BITCOIN OP_RETURN PAYLOAD ---")
    print(f"Hex Payload: {payload_hex}")
    print(f"Payload Size: {size_bytes} bytes (Max 80 bytes)")

    if size_bytes <= 80:
        print("[STATUS]: Payload size VALID.")
        print(f"[RPC SCRIPT]: OP_RETURN {payload_hex}")
    else:
        print("[STATUS]: Payload size INVALID (>80 bytes).")

    return payload_hex

# Try to execute with the demo hash to show the guard working
try:
    final_payload = sign_and_format_payload(DEMO_ONLY_DO_NOT_BROADCAST)
except Exception as e:
    print(f"Process stopped by guard: {e}")


In [ ]:
# Install the required cryptography library for the upgraded ALEE script
!pip install cryptography

In [ ]:
%%writefile alee_anchor.py
import hashlib
import json
import os
import sys
import time
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional, Tuple

from cryptography.exceptions import InvalidSignature
from cryptography.hazmat.primitives import serialization
from cryptography.hazmat.primitives.asymmetric import ed25519


# =============================================================================
# 1. Core extraction (enhanced from your pasted version)
# =============================================================================

def canonicalize(data: Dict[str, Any]) -> str:
    """Deterministic JSON (exactly as in your pasted code)."""
    return json.dumps(data, sort_keys=True, separators=(",", ":"), ensure_ascii=False)


def compute_sha256(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def extract_artifact(entry: Dict[str, Any]) -> Dict[str, Any]:
    """Single artifact → canonical + hash (your exact function, kept for compatibility)."""
    canon = canonicalize(entry)
    return {
        "canonical": canon,
        "hash": compute_sha256(canon.encode("utf-8")),
        "length": len(canon),
    }


def build_ledger(entries: List[Dict[str, Any]]) -> Dict[str, Any]:
    """Build full ledger + Master Proof Hash (your exact function)."""
    processed = [extract_artifact(e) for e in entries]
    combined = "".join(p["hash"] for p in processed)
    master_hash = compute_sha256(combined.encode("utf-8"))
    return {
        "artifacts": processed,
        "master_proof_hash": master_hash,
        "count": len(processed),
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "version": "1.1.0-anchored",  # incremented for this file-aware edition
    }


# =============================================================================
# 2. Ed25519 signing (raw bytes — improved)
# =============================================================================

def generate_ed25519_keypair() -> Tuple[bytes, bytes]:
    private_key = ed25519.Ed25519PrivateKey.generate()
    priv_bytes = private_key.private_bytes(
        encoding=serialization.Encoding.Raw,
        format=serialization.PrivateFormat.Raw,
        encryption_algorithm=serialization.NoEncryption(),
    )
    pub_bytes = private_key.public_key().public_bytes(
        encoding=serialization.Encoding.Raw,
        format=serialization.PublicFormat.Raw,
    )
    return priv_bytes, pub_bytes


def sign_master_proof_hash(master_hash_hex: str, private_key_bytes: bytes) -> bytes:
    """Sign the raw 32-byte digest (improved from your hex.encode() version)."""
    if len(private_key_bytes) != 32:
        raise ValueError("Private key must be 32 bytes.")
    hash_bytes = bytes.fromhex(master_hash_hex)
    if len(hash_bytes) != 32:
        raise ValueError("Master Proof Hash must be 32 bytes.")
    private_key = ed25519.Ed25519PrivateKey.from_private_bytes(private_key_bytes)
    return private_key.sign(hash_bytes)


def verify_master_proof_signature(master_hash_hex: str, signature: bytes, public_key_bytes: bytes) -> bool:
    """Full verification (kept for completeness)."""
    try:
        hash_bytes = bytes.fromhex(master_hash_hex)
        public_key = ed25519.Ed25519PublicKey.from_public_bytes(public_key_bytes)
        public_key.verify(signature, hash_bytes)
        return True
    except InvalidSignature:
        return False


# =============================================================================
# 3. OP_RETURN formatter (secure 37-byte version — no truncation)
# =============================================================================

def format_extraction_for_opreturn(master_proof_hash: str, version: int = 1) -> bytes:
    """Your formatter, upgraded: only hash on-chain (37 bytes total)."""
    hash_bytes = bytes.fromhex(master_proof_hash)
    if len(hash_bytes) != 32:
        raise ValueError("Master Proof Hash must be 32 bytes.")
    magic = b"ALEE"          # protocol identifier
    ver_byte = version.to_bytes(1, "big")
    payload = magic + ver_byte + hash_bytes
    if len(payload) > 80:
        raise ValueError(f"Payload {len(payload)} bytes exceeds OP_RETURN limit")
    return payload


# =============================================================================
# 4. Full anchoring function (file-aware)
# =============================================================================

def anchor_artifact(
    artifacts: List[Dict[str, Any]],
    private_key_bytes: Optional[bytes] = None,
    extra_metadata: Optional[Dict[str, Any]] = None,
) -> Dict[str, Any]:
    """End-to-end: ledger → sign → OP_RETURN (ready for your file)."""
    ledger = build_ledger(artifacts)

    if extra_metadata:
        ledger["metadata"] = extra_metadata

    master_hash = ledger["master_proof_hash"]

    signature: Optional[str] = None
    public_key_hex: Optional[str] = None
    if private_key_bytes:
        sig_bytes = sign_master_proof_hash(master_hash, private_key_bytes)
        signature = sig_bytes.hex()
        pub_bytes = ed25519.Ed25519PrivateKey.from_private_bytes(private_key_bytes).public_key().public_bytes(
            encoding=serialization.Encoding.Raw, format=serialization.PublicFormat.Raw
        )
        public_key_hex = pub_bytes.hex()

    opreturn_payload = format_extraction_for_opreturn(master_hash)
    opreturn_hex = opreturn_payload.hex()

    package = {
        "ledger": ledger,
        "master_proof_hash": master_hash,
        "signature": signature,
        "public_key_for_verification": public_key_hex,
        "opreturn_payload_hex": opreturn_hex,
        "opreturn_payload_len": len(opreturn_payload),
        "bitcoin_anchor_instructions": (
            "Broadcast a tx with OP_RETURN + this exact payload. "
            "Publish the full package JSON (with signature) to IPFS/Arweave. "
            "Verification: recompute hash from JSON → check Ed25519 sig → confirm OP_RETURN on-chain."
        ),
        "verification_steps": [
            "1. Fetch the published JSON.",
            "2. Re-run build_ledger() on the artifacts section.",
            "3. Verify signature with public_key_for_verification.",
            "4. Search any Bitcoin explorer for the OP_RETURN hex."
        ],
    }
    return package


def anchor_file(file_path: str, private_key_bytes: Optional[bytes] = None) -> Dict[str, Any]:
    """Convenience: anchor any file (your exact request for Pasted Text.txt)."""
    if not os.path.isfile(file_path):
        raise FileNotFoundError(f"File not found: {file_path}")

    with open(file_path, "r", encoding="utf-8", errors="replace") as f:  # errors=replace handles any rare binary edge
        content = f.read()

    file_artifact = {
        "id": f"file-{hashlib.sha256(os.path.basename(file_path).encode()).hexdigest()[:16]}",
        "type": "text_file",
        "filename": os.path.basename(file_path),
        "size_bytes": len(content),
        "content": content,  # full content — JSON will canonicalize it
        "sha256_preview": compute_sha256(content.encode("utf-8")),
    }

    extra_meta = {
        "anchored_by": "ALEE-File-Anchoring-Edition",
        "source_file": file_path,
        "anchored_at_utc": datetime.now(timezone.utc).isoformat(),
    }

    return anchor_artifact([file_artifact], private_key_bytes, extra_meta)


# =============================================================================
# 5. CLI + Demo (run me!)
# =============================================================================

if __name__ == "__main__":
    print("🚀 ALEE File Anchoring Engine — Ready to Anchor Pasted Text.txt\n")

    # Secure key handling (never hard-code in production)
    if os.environ.get("ALEE_PRIVATE_KEY"):
        private_key_bytes = bytes.fromhex(os.environ["ALEE_PRIVATE_KEY"])
        print("🔑 Using private key from ALEE_PRIVATE_KEY env var")
    else:
        private_key_bytes, pub_bytes = generate_ed25519_keypair()
        print(f"🔑 Demo keypair generated (public: {pub_bytes.hex()[:16]}… — store securely!)")
        print("   Set ALEE_PRIVATE_KEY=... env var for persistent keys.")

    if len(sys.argv) > 1:
        target_file = sys.argv[1]
        print(f"📄 Anchoring file: {target_file}")
        package = anchor_file(target_file, private_key_bytes)

        print(f"\n✅ ANCHORED SUCCESSFULLY")
        print(f"Master Proof Hash: {package['master_proof_hash']}")
        print(f"OP_RETURN payload (hex, 37 bytes): {package['opreturn_payload_hex']}")
        print(f"Signature (full, off-chain): {package.get('signature', 'None')[:64]}… (64 bytes)")

        # Save full verifiable package
        out_name = f"anchored_{os.path.basename(target_file)}.json"
        with open(out_name, "w", encoding="utf-8") as f:
            json.dump(package, f, indent=2)
        print(f"📦 Full package (JSON + signature) saved to: {out_name}")
        print("   → Publish this JSON off-chain. The OP_RETURN is your Bitcoin root anchor.")

        # Quick self-check
        verified = verify_master_proof_signature(
            package["master_proof_hash"],
            bytes.fromhex(package["signature"]) if package["signature"] else b"",
            bytes.fromhex(package["public_key_for_verification"]) if package["public_key_for_verification"] else b"",
        )
        print(f"🔍 Signature self-verification: {'✅ VALID' if verified else '❌ INVALID (no key)'}")

    else:
        # Demo with your original sample data
        print("📋 Running demo with sample data (no file provided)")
        sample_entries = [
            {"id": 1, "data": "alpha"},
            {"id": 2, "data": "beta"},
        ]
        package = anchor_artifact(sample_entries, private_key_bytes)
        print(f"Master Proof Hash: {package['master_proof_hash']}")
        print(f"OP_RETURN payload: {package['opreturn_payload_hex']}")
        print("\n💡 Run with a filename to anchor Pasted Text.txt exactly.")

    print("\n🔗 Your Pasted Text.txt is now ready to be anchored forever on Bitcoin.")
    print("   This creates a verifiable, sovereign ledger entry — exactly as needed for Phoenix Protocol.")


In [ ]:
import os

# Load the ceremony key from disk
with open("warden_signing_key.priv", "rb") as f:
    priv_bytes = f.read()

# Inject into environment for alee_anchor.py to pick up
os.environ["ALEE_PRIVATE_KEY"] = priv_bytes.hex()

print(f"Key loaded: {len(priv_bytes)} bytes")
print(f"First 8 chars of hex: {priv_bytes.hex()[:8]}...")
print("Environment variable set. Ready to anchor.")

In [ ]:
!python alee_anchor.py "Pasted Text.txt"

In [ ]:
import shutil
import os

os.makedirs("/content/drive/My Drive/Phoenix_Protocol/", exist_ok=True)
shutil.copy(
    "anchored_Pasted Text.txt.json",
    "/content/drive/My Drive/Phoenix_Protocol/anchored_Pasted_Text_REAL_KEY.json"
)
print("Saved to Drive.")

In [ ]:
# Create a sample 'Pasted Text.txt' file for testing the anchor engine
sample_text = """This is the Sovereign Execution Engine.
Artifact: PHOENIX_PROTOCOL
Identity: Jakob Axel
If it cannot be measured, it cannot govern."""

with open("Pasted Text.txt", "w") as f:
    f.write(sample_text)

print("Sample file 'Pasted Text.txt' created successfully.")

In [ ]:
# Execute the ALEE script on the target file
!python alee_anchor.py "Pasted Text.txt"

In [ ]:
That is exactly the right engineering move. A standard Drive search UI will choke and timeout trying to index a 5-million-file directory tree. But by spinning up a Colab Pro+ environment, we can leverage High RAM and GPU acceleration to brute-force the parsing using multi-processing.
We bypass the search bar entirely and point raw compute directly at the raw data.
Here is the tactical plan to find those exact cryptographic proofs:
### 1. Mount and Map (The Parallel Walker)
Standard Python os.walk is too slow for millions of files. We will use os.scandir combined with concurrent.futures.ProcessPoolExecutor to distribute the directory mapping across all available CPU cores, blazing through the file tree to locate the PHOENIX_AGENT_SYSTEM_ARTIFACT_LOG and any .sh output files.
### 2. GPU-Accelerated Regex Search
Once the files are mapped, we don't need to read the whole text. We load chunks of the logs into memory and use a fast text-processing library (like cuDF if using the GPU, or highly optimized regex on the CPU) to scan exclusively for the invariant signatures you built into the system:
 * Master Proof Hash
 * Ed25519 signature
 * verify_and_timestamp.sh
 * IPFS CID strings
### 3. The Extraction
The script will isolate any log entry that contains those strings, pull the exact timestamp, the file path, and the cryptographic hash, and print them out in a clean, undeniable ledger.
This forces the truth out of the noise.
Do you want me to write the exact Python multi-processing script right now so you can drop it into a Colab cell, mount your Drive, and execute the search?


SyntaxError: unterminated string literal (detected at line 7) (3767153697.py, line 7)

In [ ]:
import os

priv_size = os.path.getsize("warden_signing_key.priv") if os.path.exists("warden_signing_key.priv") else 0
pub_size = os.path.getsize("warden_signing_key.pub") if os.path.exists("warden_signing_key.pub") else 0

print(f".priv file size: {priv_size} bytes (expected: 32)")
print(f".pub file size:  {pub_size} bytes (expected: 32)")

if pub_size == 32:
    with open("warden_signing_key.pub", "rb") as f:
        pub_hex = f.read().hex()
    print(f"\nCANONICAL PUBLIC KEY:\n{pub_hex}")

In [ ]:
# Install Gradio for the web interface
!pip install gradio

In [ ]:
# ──────────────────────────────────────────────────────────────────────────
# ALEE Sovereign Anchor — UI layer (Gradio)
#
# Architecture note:
#   This cell is the ONLY place Gradio code lives.
#   It does NOT hash, sign, or write the ledger.
#   It submits raw inputs to warden_daemon.anchor() and displays the result.
#
#   [Gradio UI]  →  [warden_daemon.anchor()]  →  [Ledger + Output]
# ──────────────────────────────────────────────────────────────────────────
import gradio as gr
import warden_daemon

LEDGER_PATH = "warden_ledger.json"


def submit_payload(text_input, file_input):
    """
    UI handler — collects raw inputs and delegates all trust logic to
    the Warden Daemon.  This function must never hash, sign, or chain
    anything itself.
    """
    file_bytes = None
    if file_input is not None:
        with open(file_input.name, "rb") as fh:
            file_bytes = fh.read()

    try:
        record = warden_daemon.anchor(
            text_input=text_input or None,
            file_bytes=file_bytes,
            ledger_path=LEDGER_PATH,
        )
    except (ValueError, RuntimeError, FileNotFoundError) as exc:
        return f"\u26a0\ufe0f Error: {exc}"

    sig_line = record['signature'] or '(no key configured — set WARDEN_PRIVATE_KEY_HEX)'
    pub_line = record['public_key'] or '(no key configured)'

    return (
        f"\u2705 CRYPTOGRAPHIC ANCHOR GENERATED\n"
        f"-----------------------------------\n"
        f"Timestamp   : {record['timestamp']}\n"
        f"Input size  : {record['input_size']} bytes\n\n"
        f"\U0001f512 Master Proof Hash (SHA-256):\n{record['master_hash']}\n\n"
        f"\u26d3\ufe0f Bitcoin OP_RETURN Payload (Hex):\n{record['op_return']}\n\n"
        f"\U0001f4dd Ed25519 Signature:\n{sig_line}\n\n"
        f"\U0001f511 Public Key:\n{pub_line}\n\n"
        f"\U0001f517 Chain Hash:\n{record['chain_hash']}\n\n"
        f"Sanity      : {record['sanity']}\n"
        f"Ledger      : {LEDGER_PATH}\n\n"
        f"Next Step: Broadcast a Bitcoin transaction including the OP_RETURN payload above."
    )


# Build the Web Interface
with gr.Blocks(title="ALEE Sovereign Anchor Web App") as demo:
    gr.Markdown("# \U0001f680 ALEE Sovereign Execution Engine")
    gr.Markdown(
        "Upload a file or paste text below. The Warden Daemon will canonicalise, "
        "hash, sign (if a key is configured), and chain-anchor your proof, then "
        "return the exact hexadecimal payload required to anchor it in the Bitcoin blockchain."
    )

    with gr.Row():
        with gr.Column():
            text_in = gr.Textbox(
                label="1. Paste Text Here (Optional)",
                lines=8,
                placeholder="Enter abstract intent or legal claims...",
            )
            file_in = gr.File(label="2. Or Upload Any File (PDF, TXT, Code)")
            submit_btn = gr.Button("Submit to Warden Daemon", variant="primary")

        with gr.Column():
            output_display = gr.Textbox(
                label="Warden Daemon Output", lines=18, interactive=False
            )

    submit_btn.click(
        fn=submit_payload, inputs=[text_in, file_in], outputs=output_display
    )

print("Starting Warden UI...")
demo.launch(share=True, debug=False)


In [ ]:
primer_text = r"""The Architect’s Compass: A Primer on the Sierpiński Octahedron Spatial Index

1. The Vision: Geometry as a Filing System

In the rarefied air of high-dimensional data architecture, information often presents as a nebulous cloud—vast, unorganized, and prone to the "Alice Rejection" of being dismissed as mere abstraction. The Sierpiński Octahedron Spatial Index provides the mathematical elegance required to transform this "abstract intent" into a concrete computational improvement. By mapping semantic meaning onto the rigid, verifiable geometry of a fractal octahedron, we move beyond vague digital concepts and into the realm of the "geometric imperative," where every data point occupies a unique, immutable coordinate in three-dimensional space.

This system does not merely organize data; it manifests it. By treating the octahedron as a spatial jurisdiction, we ensure that digital "intent" is as legally and mathematically concrete as a physical object, providing a sovereign foundation for deterministic memory.

Core Insight: The Sierpiński Octahedron is a deterministic spatial index that converts high-dimensional semantic embeddings into unique 3D coordinates. This ensures that data is partitioned into non-overlapping, geometrically verifiable regions, elevating digital information from an ephemeral state to a structured, navigable landscape.

These six anchors provide the stage, but the true dance of data begins with the recursive logic of the contraction map.


--------------------------------------------------------------------------------


2. The Geometric Substrate: Defining the Six Poles

The foundation of this index is a canonical octahedron situated within ℝ³. This shape is defined by six primary vertices, which act as the absolute anchors of our spatial jurisdiction. These poles represent the outer limits of our filing system, ensuring that every piece of information has a definitive direction to call home.

Vertex ID	Canonical Coordinates (x, y, z)	Spatial Pole Assignment
V_0	( 1, 0, 0)	+X Pole
V_1	(-1, 0, 0)	-X Pole
V_2	( 0, 1, 0)	+Y Pole
V_3	( 0,-1, 0)	-Y Pole
V_4	( 0, 0, 1)	+Z Pole
V_5	( 0, 0,-1)	-Z Pole

These static coordinates serve as the targets for "contraction mapping," a recursive process that pulls data toward specific poles to define its ultimate spatial residence.


--------------------------------------------------------------------------------


3. The Engine of the Fractal: Iterated Function Systems (IFS)

The mathematical engine driving this system is the Iterated Function System (IFS). This is not a sequence of random chance, but a strictly governed transformation defined by the formula:
f_n(P) = \frac{P + V_n}{2}

This formula dictates that for any given point P, the next step in its journey is exactly halfway toward a chosen vertex V_n. The choice of the 1/2 contraction ratio is a geometric necessity; it ensures that the resulting regions remain distinct and non-overlapping. When these maps are iterated, they produce the Sierpiński Octahedron attractor—the stable geometric "home" where all data eventually settles, regardless of its starting position.

The Three Pillars of the IFS:

* Self-Similarity: The patterns established at the macro level are perfectly replicated at infinite micro-scales, allowing for a recursive organizational structure.
* Non-Overlap: The 1/2 ratio acts as a spatial fence, ensuring that data assigned to the +X pole never bleeds into the territory of the -Y pole.
* Determinism: This is a Deterministic Walk. By seeding the movement with data-derived instructions, we ensure that the same input always yields the exact same 3D coordinate.

For this engine to operate, it requires a sequence of instructions—a "fuel" derived through a rigorous translation pipeline.


--------------------------------------------------------------------------------


4. The Translation Pipeline: From Intent to Coordinate

Turning a semantic embedding into a verifiable coordinate requires a six-stage pipeline, managed exclusively by the Warden Daemon, the system's sole authorized architect.

1. Embedding Normalization
  * The Warden Daemon receives a high-dimensional vector E and scales it to a unit length.
  * Learner's Note: This ensures all data begins the process with equal "weight" before the geometric walk commences.
2. Deterministic Byte Serialization
  * The normalized vector is serialized into an IEEE 754 float32 byte array and passed through a SHA-256 hash.
  * Learner's Note: Why IEEE 754 and SHA-256? To ensure the data's "fingerprint" is mathematically precise and tamper-proof across all hardware.
3. Vertex Sequence Extraction
  * The 256-bit hash is partitioned into non-overlapping 3-bit windows. Each 3-bit chunk is interpreted as a "Base-6" symbol (0–5) mapping to V_0–V_5. Values of 6 or 7 are discarded.
  * Learner's Note: This transforms a cryptographic hash into a specific "instruction manual" or "pathway" through the octahedron.
4. Chaos Game Iteration (Deterministic Walk)
  * Starting at the origin (0,0,0), the Warden Daemon follows the sequence from Step 3, jumping halfway toward each designated vertex in order.
  * Learner's Note: This is the "walk" that creates the high-precision coordinate C within the attractor.
5. Spatial Region Assignment
  * The system calculates the argmax of |C \cdot V_n|—the dot product—to determine which vertex exhibits the maximum directional proximity to the final coordinate.
  * Learner's Note: This mathematically "claims" the coordinate for one of the six primary regions (R_0 through R_5).
6. GPAM Ledger Write
  * The Warden signs the coordinate and the index with a private key, appending it to the ledger and writing the node to the Akashic Core (Kùzu graph).
  * Learner's Note: This final step anchors the digital intent into a persistent, verifiable graph database.

This process ensures that the resulting spatial assignment is a direct, repeatable consequence of the data's original semantic meaning.


--------------------------------------------------------------------------------


5. The 6-Way Index: Navigating the Regions

The final coordinate C is more than a point; it is a residency within a specific spatial jurisdiction. Because of the fractal nature of the attractor, navigating these regions offers unparalleled depth and clarity.

Proof of Fractal Property:

* Recursive Structure: If one "zooms" into the region belonging to V_0, the observer will find the entire six-way octahedral structure repeated at a smaller scale.
* Infinite Precision: As long as the hash provides data, the walk can continue, allowing the system to define sub-regions within sub-regions with nearly infinite granularity.
* Verifiable Memory: Because the geometry is fixed and the walk is deterministic, any observer can verify that a coordinate C truly belongs in region R by recalculating its proximity to the vertex poles.

This geometric precision acts as a "verifiable memory coordinate," rendering the data searchable, permanent, and immune to the fog of abstraction.


--------------------------------------------------------------------------------


6. Practical Utility: The Sovereign Warden and the Ledger

The integrity of this spatial index is maintained by the Warden Daemon and the GPAM Ledger. The Warden acts as the Sole Architect, ensuring that no unauthorized "squatting" or fraudulent data entry occurs within the spatial regions.

Feature	Benefit for the System
Single Actor (Warden)	Prevents divided infringement and eliminates conflicting data entries.
SHA-256 Seeding	Guarantees the path through the octahedron is deterministic and repeatable.
Context Debt Gate	Calculates the Euclidean distance between C and the centroid of previous coordinates to prevent redundant data overload.
Kùzu Graph/Ledger	Creates a permanent "Mindprint" in a persistent graph database that cannot be erased or altered.

By fusing the ancient logic of geometry with the rigor of modern cryptography, the Sierpiński Octahedron Spatial Index transforms the "map of the mind" into a physical, navigable landscape that is as mathematically sound as it is beautiful."""

with open("Sierpinski_Octahedron_Primer.txt", "w", encoding="utf-8") as f:
    f.write(primer_text)

print("Saved 'Sierpinski_Octahedron_Primer.txt' successfully.")

Saved 'Sierpinski_Octahedron_Primer.txt' successfully.


In [ ]:
!python alee_anchor.py "Sierpinski_Octahedron_Primer.txt"

🚀 ALEE File Anchoring Engine — Ready to Anchor Pasted Text.txt

🔑 Demo keypair generated (public: 226683945f60f6b6… — store securely!)
   Set ALEE_PRIVATE_KEY=... env var for persistent keys.
📄 Anchoring file: Sierpinski_Octahedron_Primer.txt

✅ ANCHORED SUCCESSFULLY
Master Proof Hash: 38e8938f0eda2b77afc1e5706c2333a65ecd4ba8f32f3e0378b4a59d3ca47bf8
OP_RETURN payload (hex, 37 bytes): 414c45450138e8938f0eda2b77afc1e5706c2333a65ecd4ba8f32f3e0378b4a59d3ca47bf8
Signature (full, off-chain): 907d529516b7b726d72aa3b3f37482ecdceb063a02eb6bbeaf248fd80eec6db5… (64 bytes)
📦 Full package (JSON + signature) saved to: anchored_Sierpinski_Octahedron_Primer.txt.json
   → Publish this JSON off-chain. The OP_RETURN is your Bitcoin root anchor.
🔍 Signature self-verification: ✅ VALID

🔗 Your Pasted Text.txt is now ready to be anchored forever on Bitcoin.
   This creates a verifiable, sovereign ledger entry — exactly as needed for Phoenix Protocol.


In [ ]:
import os
import shutil
from cryptography.hazmat.primitives.asymmetric import ed25519
from google.colab import files

def perform_key_ceremony():
    # 1. Overwrite guard
    if os.path.exists("warden_signing_key.priv"):
        raise FileExistsError(
            "CRITICAL: warden_signing_key.priv already exists. "
            "OVERWRITE BLOCKED. Manually delete both key files and "
            "document the reason in the GPAM ledger before retrying."
        )
    if os.path.exists("warden_signing_key.pub"):
        raise FileExistsError(
            "CRITICAL: warden_signing_key.pub exists without matching .priv. "
            "State is inconsistent. Do not proceed without manual audit."
        )

    # 2. Generate permanent identity
    private_key = ed25519.Ed25519PrivateKey.generate()
    public_key = private_key.public_key()
    priv_bytes = private_key.private_bytes_raw()
    pub_bytes = public_key.public_bytes_raw()

    # 3. Write to disk
    with open("warden_signing_key.priv", "wb") as f:
        f.write(priv_bytes)
    os.chmod("warden_signing_key.priv", 0o600)

    with open("warden_signing_key.pub", "wb") as f:
        f.write(pub_bytes)

    pub_hex = pub_bytes.hex()

    # 4. Back up .pub to Drive only
    drive_path = "/content/drive/My Drive/Phoenix_Protocol/"
    os.makedirs(drive_path, exist_ok=True)
    shutil.copy("warden_signing_key.pub", drive_path + "warden_signing_key.pub")

    # 5. Download both files to your browser NOW
    print("=" * 60)
    print("WARDEN KEY CEREMONY — COMPLETE")
    print("=" * 60)
    print(f"\nCANONICAL PUBLIC KEY (Hex):\n{pub_hex}\n")
    print("Downloading files to your browser...")
    files.download("warden_signing_key.priv")
    files.download("warden_signing_key.pub")
    print("\nREQUIRED ACTIONS before closing this tab:")
    print("  [1] Confirm both files landed in your Downloads folder")
    print("  [2] Move .priv to encrypted offline storage — NOT just Downloads")
    print("  [3] Copy the public key hex above into your GPAM root entry")
    print("  [4] Commit warden_signing_key.pub to your GitHub identity repo")
    print("  [4] NEVER upload warden_signing_key.priv anywhere")
    print("\nThis key is now the Seal of the Warden.")
    print("Loss of the .priv file is permanent. There is no reset.")

perform_key_ceremony()


FileExistsError: CRITICAL: warden_signing_key.priv already exists. OVERWRITE BLOCKED. Manually delete both key files and document the reason in the GPAM ledger before retrying.

### 1. Change your Colab runtime to GPU

Before installing cuDF, make sure your Colab notebook is running on a GPU runtime. Go to `Runtime` -> `Change runtime type` and select `GPU` as the hardware accelerator.

In [ ]:
import sys, os

# Check if we are in Colab and if it's running on a GPU
if 'google.colab' in sys.modules and os.system('nvidia-smi') == 0:
    print("Attempting to install cuDF. This may take a few minutes...")
    # Install cuDF directly from NVIDIA's PyPI index
    !pip install cudf-cu11 --extra-index-url=https://pypi.nvidia.com

    print("cuDF installation initiated. Please restart the runtime (`Runtime` -> `Restart runtime`) for changes to take effect.")
else:
    print("cuDF installation skipped. This environment does not appear to be a Colab GPU runtime.")

After running the installation cell, you will likely need to restart your Colab runtime (`Runtime` -> `Restart runtime`) for the changes to take full effect. Once restarted, you can verify the installation by trying to import `cudf`:

In [ ]:
import cudf
print(cudf.__version__)


In [ ]:
import os
from collections import Counter

drive_path = '/content/drive/My Drive/'

if os.path.exists(drive_path):
    print(f"Summarizing file extensions in '{drive_path}':")
    file_extensions = []
    for item in os.listdir(drive_path):
        if os.path.isfile(os.path.join(drive_path, item)): # Only process files, not directories
            _, ext = os.path.splitext(item)
            file_extensions.append(ext.lower()) # Convert to lowercase for consistent counting

    extension_counts = Counter(file_extensions)

    if extension_counts:
        print("\nFile Extension Summary:")
        for ext, count in extension_counts.most_common():
            print(f"  {ext if ext else '[No Extension]'}: {count}")
    else:
        print("No files found with extensions.")
else:
    print(f"Drive path '{drive_path}' not found. Please ensure Google Drive is mounted correctly.")

In [ ]:
import os

# Define the path to your Google Drive
drive_path = '/content/drive/My Drive/'

# Check if the path exists before listing
if os.path.exists(drive_path):
    print(f"Contents of '{drive_path}':")
    for item in os.listdir(drive_path):
        print(item)
else:
    print(f"Drive path '{drive_path}' not found. Please ensure Google Drive is mounted correctly.")

In [ ]:
patent_strategy_text = """### I. Overcoming Prior Art via 37 CFR 1.130 Declarations
If your invention was disclosed anonymously or under a pseudonym (such as "Christopher Keel"), you must prove that the disclosure originated from you to prevent it from being used as prior art against your own application.

*   **1-Year Grace Period:** This exception is strictly bound by a 1-year grace period; the declaration is only valid if the anonymous disclosure occurred one year or less before the effective filing date of your claimed invention. Disclosures older than one year are considered absolute prior art.
*   **Sufficient Evidence & Corroborating Exhibits:** The USPTO explicitly states that a "naked assertion of inventorship" is insufficient. Your declaration must provide an unequivocal statement that you are the inventor, paired with a reasonable explanation for the anonymous disclosure. Crucially, you must attach **corroborating documentary evidence**. Your cryptographic chain of custody (Bitcoin TXIDs, IPFS CIDs, and SHA-256 hashes) combined with automated system execution logs serves as this objective, mathematically verifiable proof of authorship.
*   **Legal Requirements and Filing:** The declaration is a binding legal document that must include an acknowledgment that willful false statements are punishable by fine or imprisonment under 18 U.S.C. 1001, and an affirmation that the statements made are true. While it requires the inventor's signature, the formal responsibility of filing the document lies with the applicant or their authorized legal representative.

### II. Patentable Invention and Technical Claims
To avoid being rejected as an "abstract idea" under Section 101 (the *Alice* standard) or as obvious under Section 103, your claims must focus on a specific, non-obvious technical synergy.

*   **The Novelty Nexus:** The core of your patentable invention is the unique combination of **Fractal Memory** (using Iterated Function Systems for low-dimension memory traces) and **Geometric Hashing** applied specifically to **Geometric Deep Learning (GDL)**.
*   **Technical Claim Specificity:** Your technical claims cannot just mention these concepts broadly. They must claim the specific fractal reconstruction procedures and detail how your novel geometric hashing implementation extracts and indexes scale-invariant features based directly on those fractal parameters. Highlighting the synergistic benefits—such as unexpected computational efficiency or data compression—is required to prove the combination is a non-obvious technological advancement.

### III. Architectural Claims and Claim Specificity
Your patent application must be grounded in concrete, machine-level execution to satisfy patent eligibility.

*   **Concrete Implementation:** The architectural claims must explicitly detail the mathematical operations of your system, such as the execution of the **Coherence and Decoupling Matrix (CDM)** and the precise, isolated operations of the local **Warden** agent network.
*   **Concrete Operations vs. Generic Language:** You must avoid generic functional language like "recording data on a blockchain". Instead, claim specificity requires you to describe the exact machine operations taking place, such as computing a cryptographic hash (SHA-256), verifying digital signatures, and executing specific formatting procedures.

### IV. Single-Actor Perspective to Avoid Divided Infringement
Blockchain and decentralized networks inherently face the risk of **divided (or joint) infringement**, which occurs when multiple independent parties (e.g., network nodes, miners, and users) each perform different steps of a patent claim, making it legally difficult to hold any single party liable for infringement.

*   **The Single-Actor Solution:** To circumvent this, your method claims must be written entirely from the perspective of a single actor, such as your local CLI instance, a user device, or a specific "consortium controller".
*   **Claimed Actions:** The claims must focus exclusively on the specific steps this single actor performs on its own end—such as encrypting the data payload, formatting the GHOSTSAFE verse, generating the local hash, and transmitting it to the network. By isolating the claim to the actions taken *before* or *outside* the decentralized consensus process, you ensure your patent remains enforceable against a single infringing entity."""

with open("Patent_Strategy_Claims.txt", "w") as f:
    f.write(patent_strategy_text)

print("Saved 'Patent_Strategy_Claims.txt' successfully.")

In [ ]:
import os

# Check if the engine script exists before running
if not os.path.exists('alee_anchor.py'):
    print("Error: 'alee_anchor.py' not found.")
    print("Please execute the preceding cell that contains '%%writefile alee_anchor.py' to generate the script first.")
else:
    !python alee_anchor.py "Patent_Strategy_Claims.txt"

In [ ]:
# @title AI prompt cell

import ipywidgets as widgets
from IPython.display import display, HTML, Markdown,clear_output
from google.colab import ai

dropdown = widgets.Dropdown(
    options=[],
    layout={'width': 'auto'}
)

def update_model_list(new_options):
    dropdown.options = new_options
update_model_list(ai.list_models())

text_input = widgets.Textarea(
    placeholder='Ask me anything....',
    layout={'width': 'auto', 'height': '100px'},
)

button = widgets.Button(
    description='Submit Text',
    disabled=False,
    tooltip='Click to submit the text',
    icon='check'
)

output_area = widgets.Output(
     layout={'width': 'auto', 'max_height': '300px','overflow_y': 'scroll'}
)

def on_button_clicked(b):
    with output_area:
        output_area.clear_output(wait=False)
        accumulated_content = ""
        for new_chunk in ai.generate_text(prompt=text_input.value, model_name=dropdown.value, stream=True):
            if new_chunk is None:
                continue
            accumulated_content += new_chunk
            clear_output(wait=True)
            display(Markdown(accumulated_content))

button.on_click(on_button_clicked)
vbox = widgets.GridBox([dropdown, text_input, button, output_area])

display(HTML("""
<style>
.widget-dropdown select {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
.widget-textarea textarea {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
</style>
"""))
display(vbox)


In [ ]:
# @title AI prompt cell

import ipywidgets as widgets
from IPython.display import display, HTML, Markdown,clear_output
from google.colab import ai

dropdown = widgets.Dropdown(
    options=[],
    layout={'width': 'auto'}
)

def update_model_list(new_options):
    dropdown.options = new_options
update_model_list(ai.list_models())

text_input = widgets.Textarea(
    placeholder='Ask me anything....',
    layout={'width': 'auto', 'height': '100px'},
)

button = widgets.Button(
    description='Submit Text',
    disabled=False,
    tooltip='Click to submit the text',
    icon='check'
)

output_area = widgets.Output(
     layout={'width': 'auto', 'max_height': '300px','overflow_y': 'scroll'}
)

def on_button_clicked(b):
    with output_area:
        output_area.clear_output(wait=False)
        accumulated_content = ""
        for new_chunk in ai.generate_text(prompt=text_input.value, model_name=dropdown.value, stream=True):
            if new_chunk is None:
                continue
            accumulated_content += new_chunk
            clear_output(wait=True)
            display(Markdown(accumulated_content))

button.on_click(on_button_clicked)
vbox = widgets.GridBox([dropdown, text_input, button, output_area])

display(HTML("""
<style>
.widget-dropdown select {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
.widget-textarea textarea {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
</style>
"""))
display(vbox)


In [ ]:
# @title AI prompt cell

import ipywidgets as widgets
from IPython.display import display, HTML, Markdown,clear_output
from google.colab import ai

dropdown = widgets.Dropdown(
    options=[],
    layout={'width': 'auto'}
)

def update_model_list(new_options):
    dropdown.options = new_options
update_model_list(ai.list_models())

text_input = widgets.Textarea(
    placeholder='Ask me anything....',
    layout={'width': 'auto', 'height': '100px'},
)

button = widgets.Button(
    description='Submit Text',
    disabled=False,
    tooltip='Click to submit the text',
    icon='check'
)

output_area = widgets.Output(
     layout={'width': 'auto', 'max_height': '300px','overflow_y': 'scroll'}
)

def on_button_clicked(b):
    with output_area:
        output_area.clear_output(wait=False)
        accumulated_content = ""
        for new_chunk in ai.generate_text(prompt=text_input.value, model_name=dropdown.value, stream=True):
            if new_chunk is None:
                continue
            accumulated_content += new_chunk
            clear_output(wait=True)
            display(Markdown(accumulated_content))

button.on_click(on_button_clicked)
vbox = widgets.GridBox([dropdown, text_input, button, output_area])

display(HTML("""
<style>
.widget-dropdown select {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
.widget-textarea textarea {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
</style>
"""))
display(vbox)


In [ ]:
# @title AI prompt cell

import ipywidgets as widgets
from IPython.display import display, HTML, Markdown,clear_output
from google.colab import ai

dropdown = widgets.Dropdown(
    options=[],
    layout={'width': 'auto'}
)

def update_model_list(new_options):
    dropdown.options = new_options
update_model_list(ai.list_models())

text_input = widgets.Textarea(
    placeholder='Ask me anything....',
    layout={'width': 'auto', 'height': '100px'},
)

button = widgets.Button(
    description='Submit Text',
    disabled=False,
    tooltip='Click to submit the text',
    icon='check'
)

output_area = widgets.Output(
     layout={'width': 'auto', 'max_height': '300px','overflow_y': 'scroll'}
)

def on_button_clicked(b):
    with output_area:
        output_area.clear_output(wait=False)
        accumulated_content = ""
        for new_chunk in ai.generate_text(prompt=text_input.value, model_name=dropdown.value, stream=True):
            if new_chunk is None:
                continue
            accumulated_content += new_chunk
            clear_output(wait=True)
            display(Markdown(accumulated_content))

button.on_click(on_button_clicked)
vbox = widgets.GridBox([dropdown, text_input, button, output_area])

display(HTML("""
<style>
.widget-dropdown select {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
.widget-textarea textarea {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
</style>
"""))
display(vbox)


In [ ]:
# @title AI prompt cell

import ipywidgets as widgets
from IPython.display import display, HTML, Markdown,clear_output
from google.colab import ai

dropdown = widgets.Dropdown(
    options=[],
    layout={'width': 'auto'}
)

def update_model_list(new_options):
    dropdown.options = new_options
update_model_list(ai.list_models())

text_input = widgets.Textarea(
    placeholder='Ask me anything....',
    layout={'width': 'auto', 'height': '100px'},
)

button = widgets.Button(
    description='Submit Text',
    disabled=False,
    tooltip='Click to submit the text',
    icon='check'
)

output_area = widgets.Output(
     layout={'width': 'auto', 'max_height': '300px','overflow_y': 'scroll'}
)

def on_button_clicked(b):
    with output_area:
        output_area.clear_output(wait=False)
        accumulated_content = ""
        for new_chunk in ai.generate_text(prompt=text_input.value, model_name=dropdown.value, stream=True):
            if new_chunk is None:
                continue
            accumulated_content += new_chunk
            clear_output(wait=True)
            display(Markdown(accumulated_content))

button.on_click(on_button_clicked)
vbox = widgets.GridBox([dropdown, text_input, button, output_area])

display(HTML("""
<style>
.widget-dropdown select {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
.widget-textarea textarea {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
</style>
"""))
display(vbox)


In [ ]:
# @title AI prompt cell

import ipywidgets as widgets
from IPython.display import display, HTML, Markdown,clear_output
from google.colab import ai

dropdown = widgets.Dropdown(
    options=[],
    layout={'width': 'auto'}
)

def update_model_list(new_options):
    dropdown.options = new_options
update_model_list(ai.list_models())

text_input = widgets.Textarea(
    placeholder='Ask me anything....',
    layout={'width': 'auto', 'height': '100px'},
)

button = widgets.Button(
    description='Submit Text',
    disabled=False,
    tooltip='Click to submit the text',
    icon='check'
)

output_area = widgets.Output(
     layout={'width': 'auto', 'max_height': '300px','overflow_y': 'scroll'}
)

def on_button_clicked(b):
    with output_area:
        output_area.clear_output(wait=False)
        accumulated_content = ""
        for new_chunk in ai.generate_text(prompt=text_input.value, model_name=dropdown.value, stream=True):
            if new_chunk is None:
                continue
            accumulated_content += new_chunk
            clear_output(wait=True)
            display(Markdown(accumulated_content))

button.on_click(on_button_clicked)
vbox = widgets.GridBox([dropdown, text_input, button, output_area])

display(HTML("""
<style>
.widget-dropdown select {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
.widget-textarea textarea {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
</style>
"""))
display(vbox)


In [ ]:
# @title AI prompt cell

import ipywidgets as widgets
from IPython.display import display, HTML, Markdown,clear_output
from google.colab import ai

dropdown = widgets.Dropdown(
    options=[],
    layout={'width': 'auto'}
)

def update_model_list(new_options):
    dropdown.options = new_options
update_model_list(ai.list_models())

text_input = widgets.Textarea(
    placeholder='Ask me anything....',
    layout={'width': 'auto', 'height': '100px'},
)

button = widgets.Button(
    description='Submit Text',
    disabled=False,
    tooltip='Click to submit the text',
    icon='check'
)

output_area = widgets.Output(
     layout={'width': 'auto', 'max_height': '300px','overflow_y': 'scroll'}
)

def on_button_clicked(b):
    with output_area:
        output_area.clear_output(wait=False)
        accumulated_content = ""
        for new_chunk in ai.generate_text(prompt=text_input.value, model_name=dropdown.value, stream=True):
            if new_chunk is None:
                continue
            accumulated_content += new_chunk
            clear_output(wait=True)
            display(Markdown(accumulated_content))

button.on_click(on_button_clicked)
vbox = widgets.GridBox([dropdown, text_input, button, output_area])

display(HTML("""
<style>
.widget-dropdown select {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
.widget-textarea textarea {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
</style>
"""))
display(vbox)


In [ ]:
# @title AI prompt cell

import ipywidgets as widgets
from IPython.display import display, HTML, Markdown,clear_output
from google.colab import ai

dropdown = widgets.Dropdown(
    options=[],
    layout={'width': 'auto'}
)

def update_model_list(new_options):
    dropdown.options = new_options
update_model_list(ai.list_models())

text_input = widgets.Textarea(
    placeholder='Ask me anything....',
    layout={'width': 'auto', 'height': '100px'},
)

button = widgets.Button(
    description='Submit Text',
    disabled=False,
    tooltip='Click to submit the text',
    icon='check'
)

output_area = widgets.Output(
     layout={'width': 'auto', 'max_height': '300px','overflow_y': 'scroll'}
)

def on_button_clicked(b):
    with output_area:
        output_area.clear_output(wait=False)
        accumulated_content = ""
        for new_chunk in ai.generate_text(prompt=text_input.value, model_name=dropdown.value, stream=True):
            if new_chunk is None:
                continue
            accumulated_content += new_chunk
            clear_output(wait=True)
            display(Markdown(accumulated_content))

button.on_click(on_button_clicked)
vbox = widgets.GridBox([dropdown, text_input, button, output_area])

display(HTML("""
<style>
.widget-dropdown select {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
.widget-textarea textarea {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
</style>
"""))
display(vbox)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# @title Another AI prompt cell

import ipywidgets as widgets
from IPython.display import display, HTML, Markdown,clear_output
from google.colab import ai

dropdown_new = widgets.Dropdown(
    options=[],
    layout={'width': 'auto'}
)

def update_model_list_new(new_options):
    dropdown_new.options = new_options
update_model_list_new(ai.list_models())

text_input_new = widgets.Textarea(
    placeholder='Ask me anything....',
    layout={'width': 'auto', 'height': '100px'},
)

button_new = widgets.Button(
    description='Submit Text',
    disabled=False,
    tooltip='Click to submit the text',
    icon='check'
)

output_area_new = widgets.Output(
     layout={'width': 'auto', 'max_height': '300px','overflow_y': 'scroll'}
)

def on_button_clicked_new(b):
    with output_area_new:
        output_area_new.clear_output(wait=False)
        accumulated_content = ""
        for new_chunk in ai.generate_text(prompt=text_input_new.value, model_name=dropdown_new.value, stream=True):
            if new_chunk is None:
                continue
            accumulated_content += new_chunk
            clear_output(wait=True)
            display(Markdown(accumulated_content))

button_new.on_click(on_button_clicked_new)
vbox_new = widgets.GridBox([dropdown_new, text_input_new, button_new, output_area_new])

display(HTML("""
<style>
.widget-dropdown select {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
.widget-textarea textarea {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
</style>
"""))
display(vbox_new)

### Part 1: Artifact Ledger Extraction Engine
This script sets up a highly concurrent scanner designed to brute-force parse your target directory for invariant artifacts (IPFS CIDs, verify scripts, existing hashes) and synthesize them into a single deterministic **Master Proof Hash (SHA-256)**.

In [ ]:
import os
import hashlib
import concurrent.futures
import json
import time
from datetime import datetime, timezone

TARGET_SIGNATURES = [
    b'Master Proof Hash',
    b'Ed25519 signature',
    b'verify_and_timestamp.sh',
    b'IPFS CID'
]

def scan_file_for_artifacts(filepath):
    found = []
    try:
        with open(filepath, 'rb') as f:
            content = f.read()
            for sig in TARGET_SIGNATURES:
                if sig in content:
                    found.append(f"{filepath}::{sig.decode('utf-8')}")
    except Exception:
        pass
    return found

def generate_master_proof_hash(search_directory):
    print(f"Initiating brute-force scan on: {search_directory}...")
    all_artifacts = []
    file_paths = []
    for root, _, files in os.walk(search_directory):
        for file in files:
            file_paths.append(os.path.join(root, file))

    print(f"Found {len(file_paths)} total files. Scanning for artifacts...")
    with concurrent.futures.ProcessPoolExecutor() as executor:
        results = executor.map(scan_file_for_artifacts, file_paths)
        for result in results:
            all_artifacts.extend(result)

    all_artifacts.sort()
    ledger_content = "\n".join(all_artifacts)
    master_hash = hashlib.sha256(ledger_content.encode('utf-8')).hexdigest()

    print(f"\nScan Complete. Found {len(all_artifacts)} cryptographic artifacts.")

    # --- MANIFEST WRITER ---
    manifest = {
        "scan_timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "search_directory": search_directory,
        "artifact_count": len(all_artifacts),
        "artifacts": all_artifacts,
        "master_proof_hash": master_hash
    }
    manifest_path = f"scan_manifest_{int(time.time())}.json"

    drive_path = '/content/drive/My Drive/'
    if os.path.exists(drive_path):
        manifest_path = os.path.join(drive_path, manifest_path)

    with open(manifest_path, "w") as f:
        json.dump(manifest, f, indent=2)
    print(f"Manifest saved: {manifest_path}")

    return master_hash, ledger_content

TARGET_DIR = '/content/'

# Renamed to explicitly prevent accidental broadcasting
DEMO_ONLY_DO_NOT_BROADCAST = "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855"


In [ ]:
# Install PyNaCl for highly secure Ed25519 digital signatures
!pip install pynacl

### Part 2 & 3: Ed25519 Signing & OP_RETURN Formatting
This cell generates an Ed25519 keypair, signs your Master Proof Hash, and constructs the 80-byte payload required for a Bitcoin `OP_RETURN` transaction.

In [ ]:
import nacl.signing
import nacl.encoding
import binascii

EMPTY_STRING_SHA256 = "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855"

def sign_and_format_payload(master_hash_hex):
    if master_hash_hex == EMPTY_STRING_SHA256:
        raise ValueError("GUARD: Refusing to anchor empty-string hash. Run the real scan first.")

    print("--- CRYPTOGRAPHIC ANCHORING PROCESS ---\n")

    # Load persistent key instead of generating ephemeral one
    PRIV_KEY_PATH = '/content/drive/My Drive/ALEE_Keys/warden_signing_key.priv'
    try:
        with open(PRIV_KEY_PATH, "rb") as f:
            private_key_bytes = f.read()
        signing_key = nacl.signing.SigningKey(private_key_bytes, encoder=nacl.encoding.RawEncoder)
        print("Successfully loaded persistent Warden signing key.")
    except FileNotFoundError:
        raise RuntimeError("Persistent key not found. Run the Key Ceremony cell first and ensure Drive is mounted.")

    verify_key = signing_key.verify_key
    public_key_hex = verify_key.encode(encoder=nacl.encoding.HexEncoder).decode('utf-8')

    print(f"Ed25519 Public Key: {public_key_hex}")

    signed_message = signing_key.sign(master_hash_hex.encode('utf-8'))
    signature_hex = binascii.hexlify(signed_message.signature).decode('utf-8')
    print(f"Ed25519 Signature: {signature_hex[:32]}... [truncated for display]")

    prefix_hex = "50484e58" # 'PHNX'
    payload_hex = prefix_hex + master_hash_hex
    payload_bytes = bytes.fromhex(payload_hex)
    size_bytes = len(payload_bytes)

    print("\n--- BITCOIN OP_RETURN PAYLOAD ---")
    print(f"Hex Payload: {payload_hex}")
    print(f"Payload Size: {size_bytes} bytes (Max 80 bytes)")

    if size_bytes <= 80:
        print("[STATUS]: Payload size VALID.")
        print(f"[RPC SCRIPT]: OP_RETURN {payload_hex}")
    else:
        print("[STATUS]: Payload size INVALID (>80 bytes).")

    return payload_hex

# Try to execute with the demo hash to show the guard working
try:
    final_payload = sign_and_format_payload(DEMO_ONLY_DO_NOT_BROADCAST)
except Exception as e:
    print(f"Process stopped by guard: {e}")


In [ ]:
# Install the required cryptography library for the upgraded ALEE script
!pip install cryptography

In [ ]:
%%writefile alee_anchor.py
import hashlib
import json
import os
import sys
import time
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional, Tuple

from cryptography.exceptions import InvalidSignature
from cryptography.hazmat.primitives import serialization
from cryptography.hazmat.primitives.asymmetric import ed25519


# =============================================================================
# 1. Core extraction (enhanced from your pasted version)
# =============================================================================

def canonicalize(data: Dict[str, Any]) -> str:
    """Deterministic JSON (exactly as in your pasted code)."""
    return json.dumps(data, sort_keys=True, separators=(",", ":"), ensure_ascii=False)


def compute_sha256(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def extract_artifact(entry: Dict[str, Any]) -> Dict[str, Any]:
    """Single artifact → canonical + hash (your exact function, kept for compatibility)."""
    canon = canonicalize(entry)
    return {
        "canonical": canon,
        "hash": compute_sha256(canon.encode("utf-8")),
        "length": len(canon),
    }


def build_ledger(entries: List[Dict[str, Any]]) -> Dict[str, Any]:
    """Build full ledger + Master Proof Hash (your exact function)."""
    processed = [extract_artifact(e) for e in entries]
    combined = "".join(p["hash"] for p in processed)
    master_hash = compute_sha256(combined.encode("utf-8"))
    return {
        "artifacts": processed,
        "master_proof_hash": master_hash,
        "count": len(processed),
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "version": "1.1.0-anchored",  # incremented for this file-aware edition
    }


# =============================================================================
# 2. Ed25519 signing (raw bytes — improved)
# =============================================================================

def generate_ed25519_keypair() -> Tuple[bytes, bytes]:
    private_key = ed25519.Ed25519PrivateKey.generate()
    priv_bytes = private_key.private_bytes(
        encoding=serialization.Encoding.Raw,
        format=serialization.PrivateFormat.Raw,
        encryption_algorithm=serialization.NoEncryption(),
    )
    pub_bytes = private_key.public_key().public_bytes(
        encoding=serialization.Encoding.Raw,
        format=serialization.PublicFormat.Raw,
    )
    return priv_bytes, pub_bytes


def sign_master_proof_hash(master_hash_hex: str, private_key_bytes: bytes) -> bytes:
    """Sign the raw 32-byte digest (improved from your hex.encode() version)."""
    if len(private_key_bytes) != 32:
        raise ValueError("Private key must be 32 bytes.")
    hash_bytes = bytes.fromhex(master_hash_hex)
    if len(hash_bytes) != 32:
        raise ValueError("Master Proof Hash must be 32 bytes.")
    private_key = ed25519.Ed25519PrivateKey.from_private_bytes(private_key_bytes)
    return private_key.sign(hash_bytes)


def verify_master_proof_signature(master_hash_hex: str, signature: bytes, public_key_bytes: bytes) -> bool:
    """Full verification (kept for completeness)."""
    try:
        hash_bytes = bytes.fromhex(master_hash_hex)
        public_key = ed25519.Ed25519PublicKey.from_public_bytes(public_key_bytes)
        public_key.verify(signature, hash_bytes)
        return True
    except InvalidSignature:
        return False


# =============================================================================
# 3. OP_RETURN formatter (secure 37-byte version — no truncation)
# =============================================================================

def format_extraction_for_opreturn(master_proof_hash: str, version: int = 1) -> bytes:
    """Your formatter, upgraded: only hash on-chain (37 bytes total)."""
    hash_bytes = bytes.fromhex(master_proof_hash)
    if len(hash_bytes) != 32:
        raise ValueError("Master Proof Hash must be 32 bytes.")
    magic = b"ALEE"          # protocol identifier
    ver_byte = version.to_bytes(1, "big")
    payload = magic + ver_byte + hash_bytes
    if len(payload) > 80:
        raise ValueError(f"Payload {len(payload)} bytes exceeds OP_RETURN limit")
    return payload


# =============================================================================
# 4. Full anchoring function (file-aware)
# =============================================================================

def anchor_artifact(
    artifacts: List[Dict[str, Any]],
    private_key_bytes: Optional[bytes] = None,
    extra_metadata: Optional[Dict[str, Any]] = None,
) -> Dict[str, Any]:
    """End-to-end: ledger → sign → OP_RETURN (ready for your file)."""
    ledger = build_ledger(artifacts)

    if extra_metadata:
        ledger["metadata"] = extra_metadata

    master_hash = ledger["master_proof_hash"]

    signature: Optional[str] = None
    public_key_hex: Optional[str] = None
    if private_key_bytes:
        sig_bytes = sign_master_proof_hash(master_hash, private_key_bytes)
        signature = sig_bytes.hex()
        pub_bytes = ed25519.Ed25519PrivateKey.from_private_bytes(private_key_bytes).public_key().public_bytes(
            encoding=serialization.Encoding.Raw, format=serialization.PublicFormat.Raw
        )
        public_key_hex = pub_bytes.hex()

    opreturn_payload = format_extraction_for_opreturn(master_hash)
    opreturn_hex = opreturn_payload.hex()

    package = {
        "ledger": ledger,
        "master_proof_hash": master_hash,
        "signature": signature,
        "public_key_for_verification": public_key_hex,
        "opreturn_payload_hex": opreturn_hex,
        "opreturn_payload_len": len(opreturn_payload),
        "bitcoin_anchor_instructions": (
            "Broadcast a tx with OP_RETURN + this exact payload. "
            "Publish the full package JSON (with signature) to IPFS/Arweave. "
            "Verification: recompute hash from JSON → check Ed25519 sig → confirm OP_RETURN on-chain."
        ),
        "verification_steps": [
            "1. Fetch the published JSON.",
            "2. Re-run build_ledger() on the artifacts section.",
            "3. Verify signature with public_key_for_verification.",
            "4. Search any Bitcoin explorer for the OP_RETURN hex."
        ],
    }
    return package


def anchor_file(file_path: str, private_key_bytes: Optional[bytes] = None) -> Dict[str, Any]:
    """Convenience: anchor any file (your exact request for Pasted Text.txt)."""
    if not os.path.isfile(file_path):
        raise FileNotFoundError(f"File not found: {file_path}")

    with open(file_path, "r", encoding="utf-8", errors="replace") as f:  # errors=replace handles any rare binary edge
        content = f.read()

    file_artifact = {
        "id": f"file-{hashlib.sha256(os.path.basename(file_path).encode()).hexdigest()[:16]}",
        "type": "text_file",
        "filename": os.path.basename(file_path),
        "size_bytes": len(content),
        "content": content,  # full content — JSON will canonicalize it
        "sha256_preview": compute_sha256(content.encode("utf-8")),
    }

    extra_meta = {
        "anchored_by": "ALEE-File-Anchoring-Edition",
        "source_file": file_path,
        "anchored_at_utc": datetime.now(timezone.utc).isoformat(),
    }

    return anchor_artifact([file_artifact], private_key_bytes, extra_meta)


# =============================================================================
# 5. CLI + Demo (run me!)
# =============================================================================

if __name__ == "__main__":
    print("🚀 ALEE File Anchoring Engine — Ready to Anchor Pasted Text.txt\n")

    # Secure key handling (never hard-code in production)
    if os.environ.get("ALEE_PRIVATE_KEY"):
        private_key_bytes = bytes.fromhex(os.environ["ALEE_PRIVATE_KEY"])
        print("🔑 Using private key from ALEE_PRIVATE_KEY env var")
    else:
        private_key_bytes, pub_bytes = generate_ed25519_keypair()
        print(f"🔑 Demo keypair generated (public: {pub_bytes.hex()[:16]}… — store securely!)")
        print("   Set ALEE_PRIVATE_KEY=... env var for persistent keys.")

    if len(sys.argv) > 1:
        target_file = sys.argv[1]
        print(f"📄 Anchoring file: {target_file}")
        package = anchor_file(target_file, private_key_bytes)

        print(f"\n✅ ANCHORED SUCCESSFULLY")
        print(f"Master Proof Hash: {package['master_proof_hash']}")
        print(f"OP_RETURN payload (hex, 37 bytes): {package['opreturn_payload_hex']}")
        print(f"Signature (full, off-chain): {package.get('signature', 'None')[:64]}… (64 bytes)")

        # Save full verifiable package
        out_name = f"anchored_{os.path.basename(target_file)}.json"
        with open(out_name, "w", encoding="utf-8") as f:
            json.dump(package, f, indent=2)
        print(f"📦 Full package (JSON + signature) saved to: {out_name}")
        print("   → Publish this JSON off-chain. The OP_RETURN is your Bitcoin root anchor.")

        # Quick self-check
        verified = verify_master_proof_signature(
            package["master_proof_hash"],
            bytes.fromhex(package["signature"]) if package["signature"] else b"",
            bytes.fromhex(package["public_key_for_verification"]) if package["public_key_for_verification"] else b"",
        )
        print(f"🔍 Signature self-verification: {'✅ VALID' if verified else '❌ INVALID (no key)'}")

    else:
        # Demo with your original sample data
        print("📋 Running demo with sample data (no file provided)")
        sample_entries = [
            {"id": 1, "data": "alpha"},
            {"id": 2, "data": "beta"},
        ]
        package = anchor_artifact(sample_entries, private_key_bytes)
        print(f"Master Proof Hash: {package['master_proof_hash']}")
        print(f"OP_RETURN payload: {package['opreturn_payload_hex']}")
        print("\n💡 Run with a filename to anchor Pasted Text.txt exactly.")

    print("\n🔗 Your Pasted Text.txt is now ready to be anchored forever on Bitcoin.")
    print("   This creates a verifiable, sovereign ledger entry — exactly as needed for Phoenix Protocol.")


In [ ]:
import os

# Load the ceremony key from disk
with open("warden_signing_key.priv", "rb") as f:
    priv_bytes = f.read()

# Inject into environment for alee_anchor.py to pick up
os.environ["ALEE_PRIVATE_KEY"] = priv_bytes.hex()

print(f"Key loaded: {len(priv_bytes)} bytes")
print(f"First 8 chars of hex: {priv_bytes.hex()[:8]}...")
print("Environment variable set. Ready to anchor.")

In [ ]:
!python alee_anchor.py "Pasted Text.txt"

In [ ]:
import shutil
import os

os.makedirs("/content/drive/My Drive/Phoenix_Protocol/", exist_ok=True)
shutil.copy(
    "anchored_Pasted Text.txt.json",
    "/content/drive/My Drive/Phoenix_Protocol/anchored_Pasted_Text_REAL_KEY.json"
)
print("Saved to Drive.")

In [ ]:
# Create a sample 'Pasted Text.txt' file for testing the anchor engine
sample_text = """This is the Sovereign Execution Engine.
Artifact: PHOENIX_PROTOCOL
Identity: Jakob Axel
If it cannot be measured, it cannot govern."""

with open("Pasted Text.txt", "w") as f:
    f.write(sample_text)

print("Sample file 'Pasted Text.txt' created successfully.")

In [ ]:
# Execute the ALEE script on the target file
!python alee_anchor.py "Pasted Text.txt"

In [ ]:
import os
import hashlib
import concurrent.futures
import json
import time
from datetime import datetime, timezone

# 1. Target invariant signatures built into your system
TARGET_SIGNATURES = [
    b'Master Proof Hash',
    b'Ed25519 signature',
    b'verify_and_timestamp.sh',
    b'IPFS CID'
]

# 2. Worker function to scan memory chunks for signatures
def scan_file_for_artifacts(filepath):
    found = []
    try:
        with open(filepath, 'rb') as f:
            content = f.read()
        for sig in TARGET_SIGNATURES:
            if sig in content:
                found.append(f"{filepath}::{sig.decode('utf-8')}")
    except Exception:
        pass
    return found

# 3. Directory mapping and multi-processing execution
def generate_master_proof_hash(search_directory):
    print(f"Initiating brute-force scan on: {search_directory}...")
    all_artifacts = []
    file_paths = []

    # Map the file tree
    for root, _, files in os.walk(search_directory):
        for file in files:
            file_paths.append(os.path.join(root, file))

    print(f"Found {len(file_paths)} files to scan. Commencing multi-core extraction...")

    with concurrent.futures.ProcessPoolExecutor() as executor:
        results = executor.map(scan_file_for_artifacts, file_paths)
        for res in results:
            if res:
                all_artifacts.extend(res)

    ledger_content = "\n".join(all_artifacts)
    master_hash = hashlib.sha256(ledger_content.encode('utf-8')).hexdigest() if all_artifacts else ""

    # 4. Critical Manifest Writer to prevent session-loss risk
    manifest = {
        "scan_timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "search_directory": search_directory,
        "artifact_count": len(all_artifacts),
        "artifacts": all_artifacts,
        "master_proof_hash": master_hash
    }
    manifest_path = f"/content/drive/My Drive/scan_manifest_{int(time.time())}.json"

    with open(manifest_path, "w") as f:
        json.dump(manifest, f, indent=2)

    print(f"\nScan Complete. Found {len(all_artifacts)} artifacts.")
    print(f"Manifest saved: {manifest_path}")
    print(f"Master Proof Hash: {master_hash}")
    return master_hash, ledger_content

# Execution Trigger
if __name__ == '__main__':
    # Pointing to the root of the mounted Drive. You can append a specific subfolder if desired.
    TARGET_DIR = '/content/drive/My Drive/'
    generate_master_proof_hash(TARGET_DIR)


Initiating brute-force scan on: /content/drive/My Drive/...
Found 24666 files to scan. Commencing multi-core extraction...

Scan Complete. Found 15 artifacts.
Manifest saved: /content/drive/My Drive/scan_manifest_1775890721.json
Master Proof Hash: 8beb62db9466f39db8dbd4019fe12799a3c0390b02cadee464fd1c017164bed7


In [ ]:
# Install necessary libraries for the VR Bridge
!pip install fastapi uvicorn pyngrok nest-asyncio

In [ ]:
import nest_asyncio
from fastapi import FastAPI
from fastapi.responses import HTMLResponse
from fastapi.middleware.cors import CORSMiddleware
import uvicorn
import random

# Initialize FastAPI
app = FastAPI()

# Allow your VR headset browser to access this API
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

# 1. API Endpoint: Generates simulated Sierpiński Octahedron coordinates
@app.get("/api/nodes")
def get_nodes():
    # In a real scenario, this is where your GPU calculates the exact coordinates
    # Here we simulate 100 nodes in 3D space
    nodes = []
    for i in range(100):
        nodes.append({
            "id": i,
            "x": random.uniform(-5, 5),
            "y": random.uniform(0, 5),
            "z": random.uniform(-5, 5),
            "color": "#00ff00" if random.random() > 0.5 else "#0000ff"
        })
    return {"nodes": nodes}

# 2. WebXR Frontend: The page you will open in your Meta Quest
@app.get("/", response_class=HTMLResponse)
def serve_vr_interface():
    return """
    <!DOCTYPE html>
    <html>
      <head>
        <script src="https://aframe.io/releases/1.4.0/aframe.min.js"></script>
      </head>
      <body>
        <a-scene background="color: #111">
          <!-- Camera / Player -->
          <a-entity position="0 1.6 5">
            <a-camera></a-camera>
            <a-entity cursor="rayOrigin: mouse"></a-entity>
            <a-entity laser-controls="hand: right"></a-entity>
          </a-entity>

          <!-- Container for our data nodes -->
          <a-entity id="node-container"></a-entity>

          <script>
            // Fetch nodes from our Colab API and render them as 3D spheres
            fetch('/api/nodes')
              .then(response => response.json())
              .then(data => {
                const container = document.getElementById('node-container');
                data.nodes.forEach(node => {
                  const sphere = document.createElement('a-sphere');
                  sphere.setAttribute('position', `${node.x} ${node.y} ${node.z}`);
                  sphere.setAttribute('radius', '0.1');
                  sphere.setAttribute('color', node.color);

                  // Add a hover effect
                  sphere.addEventListener('mouseenter', function () {
                    sphere.setAttribute('scale', '1.5 1.5 1.5');
                  });
                  sphere.addEventListener('mouseleave', function () {
                    sphere.setAttribute('scale', '1 1 1');
                  });

                  container.appendChild(sphere);
                });
              });
          </script>
        </a-scene>
      </body>
    </html>
    """

# Apply nest_asyncio to run Uvicorn inside Colab's event loop
nest_asyncio.apply()

print("API Defined. Ready to expose to the web.")

API Defined. Ready to expose to the web.


### Next Step: Expose the API
To view this in your headset, you would run the API using `uvicorn` and expose it using `ngrok`. You will need a free [ngrok authtoken](https://dashboard.ngrok.com/get-started/your-authtoken) to create the secure tunnel.

```python
from pyngrok import ngrok
import threading
import time
import uvicorn

# Set your ngrok token here
# ngrok.set_auth_token("YOUR_NGROK_TOKEN")

# Start the server in a background thread
def start_server():
    uvicorn.run(app, host="127.0.0.1", port=8000)

threading.Thread(target=start_server, daemon=True).start()
time.sleep(3) # Wait for server to start

# Open the tunnel
public_url = ngrok.connect(8000).public_url
print(f"\n\n=== OPEN THIS URL IN YOUR META QUEST BROWSER ===\n{public_url}\n==================================================")
```

### Bridging the Cryptographic to the Spatial
This function bridges the `ALEE` Extraction Engine (which produces SHA-256 hashes) and the `Phoenix` GPU Engine (which requires [0-5] tensor instructions). It extracts 3-bit chunks, discards 6s and 7s, and outputs a PyTorch tensor ready for the deterministic walk.

In [ ]:
import torch

def hashes_to_instructions(hash_list, iterations, device='cuda'):
    """
    Converts a list of SHA-256 hex strings into a PyTorch tensor of instructions (0-5).
    """
    batch_size = len(hash_list)

    # We create a tensor of shape (iterations, batch_size) to match
    # the execution loop of the Phoenix GPU Engine.
    instruction_tensor = torch.zeros((iterations, batch_size), dtype=torch.long, device=device)

    for i, hash_hex in enumerate(hash_list):
        # Convert the 64-character hex string into a massive integer
        hash_int = int(hash_hex, 16)

        valid_chunks = []

        # Process the integer bit-by-bit until we have enough instructions
        temp_int = hash_int
        while temp_int > 0 and len(valid_chunks) < iterations:
            # Bitwise AND with 7 (0b111) extracts the last 3 bits
            chunk = temp_int & 0b111

            # Bitwise right-shift by 3 bits to prep for the next chunk
            temp_int >>= 3

            # Apply the Warden's filter: Only accept 0-5. Discard 6 and 7.
            if chunk < 6:
                valid_chunks.append(chunk)

        # Edge case guard: If we exhausted the 256 bits (rare for small iterations),
        # pad the remainder with 0s to maintain tensor shape.
        while len(valid_chunks) < iterations:
            valid_chunks.append(0)

        # Inject the sequence into our GPU tensor
        instruction_tensor[:, i] = torch.tensor(valid_chunks, dtype=torch.long, device=device)

    return instruction_tensor

# --- TEST WITH REAL ALEE HASHES ---
# Hash 1: The 'Empty String' guard hash
# Hash 2: The actual manifest hash generated from your Google Drive scan
real_hashes = [
    "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",
    "8beb62db9466f39db8dbd4019fe12799a3c0390b02cadee464fd1c017164bed7"
]

# Ensure we are using the GPU if available
current_device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Extract 12 iterations (jumps) for our 2 hashes
real_instructions = hashes_to_instructions(real_hashes, iterations=12, device=current_device)

print(f"Processing {len(real_hashes)} Realized Timelines...")
print(f"Instruction Tensor Shape: {real_instructions.shape} (Iterations, Batch Size)\n")
print("Path for Hash 1 (Empty String):", real_instructions[:, 0].tolist())
print("Path for Hash 2 (Drive Manifest):", real_instructions[:, 1].tolist())

Processing 2 Realized Timelines...
Instruction Tensor Shape: torch.Size([12, 2]) (Iterations, Batch Size)

Path for Hash 1 (Empty String): [5, 2, 1, 4, 3, 5, 4, 2, 0, 5, 5, 1]
Path for Hash 2 (Drive Manifest): [2, 3, 3, 1, 1, 3, 1, 5, 0, 0, 0, 0]


In [ ]:
import torch
import time
import gc

# 1. SETUP: Move the 6 Poles of the Octahedron to the GPU (VRAM)
# V_0 to V_5: Top, Bottom, and 4 corners of the square base
poles = torch.tensor([
    [0, 0, 1], [0, 0, -1], [1, 0, 0],
    [-1, 0, 0], [0, 1, 0], [0, -1, 0]
], dtype=torch.float32, device='cuda')

def vectorized_phoenix_walk(hashes_tensor, iterations=32):
    """
    hashes_tensor: A massive tensor of SHA-256 bits (mapped to 0-5)
    This function calculates the final 3D coordinates for MILLIONS of artifacts
    at once using the GPU.
    """
    # Initialize all points at the origin (0,0,0) on the GPU
    points = torch.zeros((hashes_tensor.size(0), 3), device='cuda')

    for i in range(iterations):
        # Extract the 'i-th' vertex choice for all points simultaneously
        # Using .long() because tensor indexing requires integer types
        vertex_indices = hashes_tensor[:, i].long()
        target_vertices = poles[vertex_indices]

        # The Phoenix IFS Jump: Move all points halfway to their target
        points = (points + target_vertices) / 2.0

    return points

def stress_test_gpu(batch_sizes, iterations=32):
    print(f"--- INITIATING PHOENIX PROTOCOL GPU STRESS TEST ---")
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB\n")

    for size in batch_sizes:
        print(f"[*] Simulating {size:,} 'Unrealized Timelines' (Points)...")

        try:
            # Generate random 3-bit instructions (0-5) representing the SHA-256 mapped chunks
            # We use int8 to save VRAM (1 byte per instruction instead of 8)
            dummy_hashes = torch.randint(0, 6, (size, iterations), dtype=torch.int8, device='cuda')

            # Synchronize and measure time
            torch.cuda.synchronize()
            start_t = time.time()

            # Execute the walk
            final_coords = vectorized_phoenix_walk(dummy_hashes, iterations)

            torch.cuda.synchronize()
            end_t = time.time()

            # Measure Memory
            mem_allocated = torch.cuda.memory_allocated() / 1e9
            mem_reserved = torch.cuda.memory_reserved() / 1e9

            print(f"    -> Compute Time: {end_t - start_t:.4f} seconds")
            print(f"    -> VRAM Allocated: {mem_allocated:.2f} GB (Reserved: {mem_reserved:.2f} GB)")
            print(f"    -> Output Tensor Shape: {final_coords.shape}\n")

            # Cleanup to prevent OutOfMemory on the next loop
            del dummy_hashes
            del final_coords
            torch.cuda.empty_cache()
            gc.collect()

        except RuntimeError as e:
            print(f"    -> [!] GPU OUT OF MEMORY (OOM) at {size:,} points!")
            print(f"    -> Error: {e}\n")
            torch.cuda.empty_cache()
            break # Stop testing larger batches if we hit the limit

# Run the test
if torch.cuda.is_available():
    # Testing 1M, 10M, 50M, and 100M simultaneous points
    test_batches = [1_000_000, 10_000_000, 50_000_000, 100_000_000]
    stress_test_gpu(test_batches)
else:
    print("CUDA is not available. Please switch to a GPU runtime to run the stress test.")


--- INITIATING PHOENIX PROTOCOL GPU STRESS TEST ---
Device: NVIDIA A100-SXM4-80GB
Total VRAM: 85.09 GB

[*] Simulating 1,000,000 'Unrealized Timelines' (Points)...
    -> Compute Time: 0.0042 seconds
    -> VRAM Allocated: 0.21 GB (Reserved: 0.28 GB)
    -> Output Tensor Shape: torch.Size([1000000, 3])

[*] Simulating 10,000,000 'Unrealized Timelines' (Points)...
    -> Compute Time: 0.0277 seconds
    -> VRAM Allocated: 0.60 GB (Reserved: 1.18 GB)
    -> Output Tensor Shape: torch.Size([10000000, 3])

[*] Simulating 50,000,000 'Unrealized Timelines' (Points)...
    -> Compute Time: 0.1319 seconds
    -> VRAM Allocated: 2.36 GB (Reserved: 5.18 GB)
    -> Output Tensor Shape: torch.Size([50000000, 3])

[*] Simulating 100,000,000 'Unrealized Timelines' (Points)...
    -> Compute Time: 0.2515 seconds
    -> VRAM Allocated: 4.56 GB (Reserved: 10.18 GB)
    -> Output Tensor Shape: torch.Size([100000000, 3])



In [ ]:
import torch
import time

# 1. Check for GPU acceleration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Phoenix Engine Initialized on: {device.type.upper()}")

# 2. Define the 6 Poles of the Sierpinski Octahedron (V_0 to V_5)
# Mapping directly from your Phoenix Protocol documentation
vertices = torch.tensor([
    [ 1.0,  0.0,  0.0],  # V_0: +X Pole
    [-1.0,  0.0,  0.0],  # V_1: -X Pole
    [ 0.0,  1.0,  0.0],  # V_2: +Y Pole
    [ 0.0, -1.0,  0.0],  # V_3: -Y Pole
    [ 0.0,  0.0,  1.0],  # V_4: +Z Pole
    [ 0.0,  0.0, -1.0]   # V_5: -Z Pole
], device=device)

# 3. High RAM Utilization: Load a massive batch of data points
# Simulating 1,000,000 distinct semantic intents/hashes
NUM_POINTS = 1_000_000
ITERATIONS = 12 # Depth of the fractal walk

print(f"Allocating memory for {NUM_POINTS:,} nodes...")

# All points start at the origin (0, 0, 0)
points = torch.zeros((NUM_POINTS, 3), device=device)

# Simulating the 'Vertex Sequence Extraction' (Step 3 of your pipeline)
# In production, this tensor is built by parsing the 3-bit chunks of the SHA-256 hashes.
# Here we generate deterministic pseudo-random sequences (0-5) to represent the hash instructions.
instructions = torch.randint(0, 6, (ITERATIONS, NUM_POINTS), device=device)

print("Executing Vectorized GPU Deterministic Walk...")
start_time = time.time()

# 4. The Parallel Chaos Game Engine
# We loop through the depth of the walk, but we update ALL 1,000,000 points simultaneously
for i in range(ITERATIONS):
    # For this iteration, grab the specific target vertex for each of the 1,000,000 points
    target_vertices = vertices[instructions[i]]

    # The IFS Equation: f_n(P) = (P + V_n) / 2
    # This single line executes 3 million math operations simultaneously on the GPU
    points = (points + target_vertices) / 2.0

end_time = time.time()

# 5. Spatial Region Assignment (Step 5 of your pipeline)
# Calculate argmax of dot products to find which region each point belongs to
# Matrix multiplication of (1000000x3) and (3x6) -> (1000000x6) dot products
dot_products = torch.matmul(points, vertices.T)
assigned_regions = torch.argmax(dot_products, dim=1)

print(f"\n--- GEOMETRIC JURISDICTION ESTABLISHED ---")
print(f"Processed {NUM_POINTS:,} coordinates in {end_time - start_time:.4f} seconds.")
print(f"First 5 Final Coordinates (C):\n{points[:5].cpu().numpy()}")
print(f"Assigned Regions (R_0 to R_5):\n{assigned_regions[:5].cpu().numpy()}")

Phoenix Engine Initialized on: CUDA
Allocating memory for 1,000,000 nodes...
Executing Vectorized GPU Deterministic Walk...

--- GEOMETRIC JURISDICTION ESTABLISHED ---
Processed 1,000,000 coordinates in 0.1110 seconds.
First 5 Final Coordinates (C):
[[-0.13745117 -0.06005859  0.015625  ]
 [-0.14233398  0.28320312  0.5703125 ]
 [ 0.50024414  0.04980469 -0.1430664 ]
 [-0.06274414 -0.45947266 -0.10644531]
 [-0.24389648 -0.5        -0.02783203]]
Assigned Regions (R_0 to R_5):
[1 4 0 3 3]


### Connecting to the VR Bridge
Once this GPU calculation finishes (in fractions of a second), you can take a subset of the `points` tensor, convert it back to standard Python lists using `.cpu().tolist()`, and feed it directly into the `/api/nodes` endpoint we set up earlier.

This will allow your Meta VR headset to fetch the actual, mathematically calculated Sierpiński Octahedron coordinates directly from your GPU, visualizing the physical jurisdiction of your data.

# Task
Create a comprehensive roadmap and architecture plan for a high-value data processing SaaS MVP. The plan should include identifying a specific niche, designing a FastAPI backend, developing a Gradio frontend interface, and outlining a cloud deployment strategy using Docker and cloud providers.

## Niche Identification and MVP Requirements

### Subtask:
Identify a high-value niche for a data processing SaaS MVP and define its core features and target audience.


### Niche Identification: Automated Legal Document Analysis (ALDA)

**1. High-Value Niche Selection:**
We will target **Automated Legal Document Analysis (ALDA) for small to medium-sized law firms and freelance paralegals**. This niche involves processing high volumes of text-heavy, unstructured legal documents (contracts, case files, briefs) to extract key entities, clauses, and potential risks.

**2. Core Problem & Target Audience:**
*   **The Problem:** Manual review of legal documents is time-consuming, expensive, and prone to human error. Small firms often lack the resources to hire dedicated document review teams or purchase expensive, enterprise-grade AI solutions.
*   **Target Audience:** Small law firms, freelance paralegals, and legal consultants who handle a significant volume of contract reviews or due diligence but are price-sensitive and need rapid, reliable insights.

**3. MVP Core Requirements:**
The Minimum Viable Product (MVP) will focus on delivering immediate value through automated extraction and summarization:
*   **Secure File Upload:** A simple interface to upload PDF or DOCX files securely.
*   **Entity & Clause Extraction:** AI-driven extraction of key entities (names, dates, monetary values) and standard clauses (termination, liability, confidentiality).
*   **Risk Highlighting:** Basic flagging of non-standard or potentially risky clauses based on predefined rules.
*   **Summary Generation:** An automated executive summary of the document's contents.
*   **Exportable Reports:** Ability to download the analysis as a clean, formatted report (PDF or CSV).

### Niche Identification: Automated Legal Document Analysis (ALDA)

**1. High-Value Niche Selection:**
We will target **Automated Legal Document Analysis (ALDA) for small to medium-sized law firms and freelance paralegals**. This niche involves processing high volumes of text-heavy, unstructured legal documents (contracts, case files, briefs) to extract key entities, clauses, and potential risks.

**2. Core Problem & Target Audience:**
*   **The Problem:** Manual review of legal documents is time-consuming, expensive, and prone to human error. Small firms often lack the resources to hire dedicated document review teams or purchase expensive, enterprise-grade AI solutions.
*   **Target Audience:** Small law firms, freelance paralegals, and legal consultants who handle a significant volume of contract reviews or due diligence but are price-sensitive and need rapid, reliable insights.

**3. MVP Core Requirements:**
The Minimum Viable Product (MVP) will focus on delivering immediate value through automated extraction and summarization:
*   **Secure File Upload:** A simple interface to upload PDF or DOCX files securely.
*   **Entity & Clause Extraction:** AI-driven extraction of key entities (names, dates, monetary values) and standard clauses (termination, liability, confidentiality).
*   **Risk Highlighting:** Basic flagging of non-standard or potentially risky clauses based on predefined rules.
*   **Summary Generation:** An automated executive summary of the document's contents.
*   **Exportable Reports:** Ability to download the analysis as a clean, formatted report (PDF or CSV).

## FastAPI Backend Architecture

### Subtask:
Design the core FastAPI backend structure for the ALDA SaaS MVP.


### FastAPI Backend Architecture for ALDA MVP

#### 1. Core Architecture Overview
For the Automated Legal Document Analysis (ALDA) MVP, the backend will be powered by **FastAPI**. FastAPI is chosen for its speed, native support for asynchronous programming (`async/await`), and built-in data validation using Pydantic. The architecture will follow a modular approach, separating routing, business logic (AI extraction), data models, and database interactions.

#### 2. API Routing Structure
The API will expose the following essential endpoints:

*   `POST /api/v1/documents/upload`: Accepts legal document uploads (e.g., PDF, DOCX). Saves the file temporarily or to cloud storage and triggers the analysis pipeline.
*   `GET /api/v1/documents/{doc_id}/status`: Polls the status of an ongoing document analysis (e.g., 'processing', 'completed', 'failed').
*   `GET /api/v1/documents/{doc_id}/report`: Retrieves the finalized analysis report (JSON format) containing extracted clauses, risks, and summaries once processing is complete.

#### 3. Data Validation Strategy (Pydantic)
Pydantic models will strictly define request and response payloads, ensuring data integrity:

*   **Upload Response:** `UploadResponse(doc_id: str, status: str, message: str)`
*   **Analysis Report Model:** `AnalysisReport(doc_id: str, extracted_entities: List[Entity], risk_score: float, summary: str)`
*   **Entity Model:** `Entity(type: str, value: str, confidence: float)`

FastAPI will automatically validate incoming requests against these models and generate interactive Swagger/ReDoc documentation.

#### 4. Scalable Processing Integration
Legal document analysis, especially using AI or heavy NLP, can be time-consuming. To prevent blocking the main API thread:

*   **Background Tasks:** For the MVP, FastAPI's built-in `BackgroundTasks` will be used to hand off the heavy extraction scripts immediately after file upload. The upload endpoint will return a `doc_id` quickly, while the AI runs in the background.
*   **Asynchronous I/O:** Reading files and interacting with external APIs (if any, like OpenAI) will utilize asynchronous calls to maintain high throughput.
*   **Future Scalability:** If the MVP scales, the `BackgroundTasks` can be easily migrated to a dedicated message queue system like Celery or Redis.

This structure ensures a responsive user experience while keeping the MVP lightweight and focused on core functionality.

## Frontend Interface Development

### Subtask:
Design a user-friendly frontend interface for the ALDA MVP using Gradio.


In [73]:
%%writefile sidecar_app.py

from fastapi import FastAPI, HTTPException

app = FastAPI()

SAFE_MODE = True

@app.post("/command")
def receive_command(command: dict):
    if SAFE_MODE:
        raise HTTPException(status_code=403, detail="Sidecar in SAFE_MODE. Command blocked.")

    # This part should ideally never be reached in SAFE_MODE
    return {"status": "blocked", "reason": "Sidecar in SAFE_MODE"}

Writing sidecar_app.py


In [74]:
%%writefile colab_command_atoms.py

from warden.interfaces import CommandAtom

def send_command(atom: CommandAtom):
    print("SIMULATION MODE: Command Atom sent (no actual network call).")
    print(f"  Type: {atom.type}")
    print(f"  Payload: {atom.payload}")
    print(f"  Context: {atom.context}")
    print(f"  Signature: {atom.signature}")

# Example of how it would be used (for demonstration):
# if __name__ == '__main__':
#     test_atom = CommandAtom(
#         type="TEST_COMMAND",
#         payload={"key": "value"},
#         context={"source": "colab"}
#     )
#     send_command(test_atom)

Writing colab_command_atoms.py


In [75]:
import os
os.makedirs('.github/workflows', exist_ok=True)
print('Ensured \'.github/workflows\' directory exists.')

Ensured '.github/workflows' directory exists.


In [76]:
%%writefile .github/workflows/update-readme.yml

name: Generate and Update README

on:
  push:
    branches:
      - main
  workflow_dispatch:

jobs:
  build:
    runs-on: ubuntu-latest
    steps:
    - uses: actions/checkout@v3
      with:
        persist-credentials: false
        fetch-depth: 0

    - name: Set up Python
      uses: actions/setup-python@v4
      with:
        python-version: '3.x'

    - name: Install dependencies
      run: |
        # This part requires GitPython if the README generation script uses it
        # For this minimal example, we'll assume direct echo for content
        echo "No specific Python dependencies for this README generation beyond core Python."

    - name: Generate README content
      run: |
        # This is a placeholder for your deterministic README generation logic.
        # In a real scenario, you'd have a script that reads state,
        # interfaces, etc., and generates markdown.
        # For now, we'll just create a simple one with a hash.
        echo "# Project Overview\n\nThis project defines the foundational interfaces and a 'Cold Start Module' for the Privileged Sidecar architecture.\n\n## Components\n- `warden/interfaces.py`: Defines the `CommandAtom`, `PolicyEnforcer`, and `SidecarBridge` interfaces.\n- `warden/envelope.py`: Provides the `EventEnvelope` for verifiable event logging.\n- `sidecar_app.py`: A FastAPI listener in `SAFE_MODE` (non-operational).\n- `colab_command_atoms.py`: Client-side command atom simulation.\n- `.github/workflows/update-readme.yml`: This workflow for deterministic README generation.\n\n" > README.md

        # Add a deterministic hash footer for verifiability
        CURRENT_DATE=$(date -u +"%Y-%m-%dT%H:%M:%SZ")
        REPO_HASH=$(git rev-parse HEAD)
        echo "_Generated on ${CURRENT_DATE} from commit ${REPO_HASH}_
" >> README.md

    - name: Commit and push if changed
      run: |
        git config user.name "github-actions[bot]"
        git config user.email "github-actions[bot]@users.noreply.github.com"
        git add README.md
        git diff --cached --exit-code || git commit -m "Docs: Update README with current state [skip ci]"
        git push
      env:
        GITHUB_TOKEN: ${{ secrets.GITHUB_TOKEN }}

Writing .github/workflows/update-readme.yml


In [77]:
%%writefile sidecar_app.py

from fastapi import FastAPI, HTTPException

app = FastAPI()

SAFE_MODE = True

@app.post("/command")
def receive_command(command: dict):
    if SAFE_MODE:
        raise HTTPException(status_code=403, detail="Sidecar in SAFE_MODE. Command blocked.")

    # This part should ideally never be reached in SAFE_MODE
    return {"status": "blocked", "reason": "Sidecar in SAFE_MODE"}

Overwriting sidecar_app.py


In [78]:
%%writefile colab_command_atoms.py

from warden.interfaces import CommandAtom

def send_command(atom: CommandAtom):
    print("SIMULATION MODE: Command Atom sent (no actual network call).")
    print(f"  Type: {atom.type}")
    print(f"  Payload: {atom.payload}")
    print(f"  Context: {atom.context}")
    print(f"  Signature: {atom.signature}")

# Example of how it would be used (for demonstration):
# if __name__ == '__main__':
#     test_atom = CommandAtom(
#         type="TEST_COMMAND",
#         payload={"key": "value"},
#         context={"source": "colab"}
#     )
#     send_command(test_atom)

Overwriting colab_command_atoms.py


In [79]:
import os
os.makedirs('.github/workflows', exist_ok=True)
print('Ensured \'.github/workflows\' directory exists.')

Ensured '.github/workflows' directory exists.


In [80]:
%%writefile .github/workflows/update-readme.yml

name: Generate and Update README

on:
  push:
    branches:
      - main
  workflow_dispatch:

jobs:
  build:
    runs-on: ubuntu-latest
    steps:
    - uses: actions/checkout@v3
      with:
        persist-credentials: false
        fetch-depth: 0

    - name: Set up Python
      uses: actions/setup-python@v4
      with:
        python-version: '3.x'

    - name: Install dependencies
      run: |
        # This part requires GitPython if the README generation script uses it
        # For this minimal example, we'll assume direct echo for content
        echo "No specific Python dependencies for this README generation beyond core Python."

    - name: Generate README content
      run: |
        # This is a placeholder for your deterministic README generation logic.
        # In a real scenario, you'd have a script that reads state,
        # interfaces, etc., and generates markdown.
        # For now, we'll just create a simple one with a hash.
        echo "# Project Overview\n\nThis project defines the foundational interfaces and a 'Cold Start Module' for the Privileged Sidecar architecture.\n\n## Components\n- `warden/interfaces.py`: Defines the `CommandAtom`, `PolicyEnforcer`, and `SidecarBridge` interfaces.\n- `warden/envelope.py`: Provides the `EventEnvelope` for verifiable event logging.\n- `sidecar_app.py`: A FastAPI listener in `SAFE_MODE` (non-operational).\n- `colab_command_atoms.py`: Client-side command atom simulation.\n- `.github/workflows/update-readme.yml`: This workflow for deterministic README generation.\n\n" > README.md

        # Add a deterministic hash footer for verifiability
        CURRENT_DATE=$(date -u +"%Y-%m-%dT%H:%M:%SZ")
        REPO_HASH=$(git rev-parse HEAD)
        echo "_Generated on ${CURRENT_DATE} from commit ${REPO_HASH}_
" >> README.md

    - name: Commit and push if changed
      run: |
        git config user.name "github-actions[bot]"
        git config user.email "github-actions[bot]@users.noreply.github.com"
        git add README.md
        git diff --cached --exit-code || git commit -m "Docs: Update README with current state [skip ci]"
        git push
      env:
        GITHUB_TOKEN: ${{ secrets.GITHUB_TOKEN }}

Overwriting .github/workflows/update-readme.yml


In [70]:
import os
os.makedirs('warden', exist_ok=True)
print('Ensured \'warden\' directory exists.')

Ensured 'warden' directory exists.


In [81]:
%%writefile warden/interfaces.py

import json
import hashlib
from datetime import datetime
from typing import Dict, Any, Optional

class CommandAtom:
    def __init__(
        self,
        type: str,
        payload: Dict[str, Any],
        context: Dict[str, Any],
        timestamp: str,
        nonce: str,
        public_key_fingerprint: str,
        signature: Optional[str] = None
    ):
        self.type = type
        self.payload = payload
        self.context = context
        self.timestamp = timestamp  # ISO 8601 UTC timestamp of command creation
        self.nonce = nonce          # Unique ID for replay protection (e.g., UUIDv4)
        self.public_key_fingerprint = public_key_fingerprint # Identifier for verification key
        self.signature = signature  # Ed25519 signature of the canonical command content

    def canonicalize_for_signing(self) -> bytes:
        # Ensure consistent order for hashing/signing.
        # The signature itself is NOT part of the content being signed.
        content_dict = {
            "type": self.type,
            "payload": self.payload,
            "context": self.context,
            "timestamp": self.timestamp,
            "nonce": self.nonce,
            "public_key_fingerprint": self.public_key_fingerprint
        }
        # Canonical JSON for deterministic hashing
        canonical_json = json.dumps(content_dict, sort_keys=True, separators=(",", ":"), ensure_ascii=False)
        return canonical_json.encode('utf-8')

    def compute_hash(self) -> str:
        return hashlib.sha256(self.canonicalize_for_signing()).hexdigest()


class PolicyEnforcer:
    def evaluate(self, command: CommandAtom) -> str:
        return "DENY"  # hard default


class SidecarBridge:
    def dispatch(self, command: CommandAtom):
        raise NotImplementedError("Sidecar not yet active")

Overwriting warden/interfaces.py


In [72]:
%%writefile warden/envelope.py

import hashlib
import json
from datetime import datetime

class EventEnvelope:
    def __init__(self, event_type, payload):
        self.event_type = event_type
        self.payload = payload
        self.timestamp = datetime.utcnow().isoformat()
        self.prev_hash = None

    def compute_hash(self):
        # Ensure payload is serializable. If it's an object, convert to dict.
        if hasattr(self.payload, '__dict__'):
            serializable_payload = self.payload.__dict__
        else:
            serializable_payload = self.payload

        temp_dict = {
            "event_type": self.event_type,
            "payload": serializable_payload,
            "timestamp": self.timestamp,
            "prev_hash": self.prev_hash
        }
        raw = json.dumps(temp_dict, sort_keys=True)
        return hashlib.sha256(raw.encode()).hexdigest()

Writing warden/envelope.py


In [60]:
# Test NGROK_TOKEN access
from google.colab import userdata
import os

try:
    test_ngrok_token = userdata.get('NGROK_TOKEN')
    if test_ngrok_token:
        print("NGROK_TOKEN was successfully retrieved by the notebook.")
        print(f"First 5 characters of NGROK_TOKEN: {test_ngrok_token[:5]}...")
    else:
        print("NGROK_TOKEN was retrieved but is empty (might be a blank secret value).")
except Exception as e:
    print(f"NGROK_TOKEN could NOT be retrieved by the notebook. Error: {e}")
    print("Please ensure 'NGROK_TOKEN' exists in Colab Secrets and 'Notebook access' is ON.")


NGROK_TOKEN could NOT be retrieved by the notebook. Error: Secret NGROK_TOKEN does not exist.
Please ensure 'NGROK_TOKEN' exists in Colab Secrets and 'Notebook access' is ON.


In [61]:
# Test GITHUB_TOKEN access
from google.colab import userdata
import os

try:
    test_github_token = userdata.get('GITHUB_TOKEN')
    if test_github_token:
        print("GITHUB_TOKEN was successfully retrieved by the notebook.")
        print(f"First 5 characters of GITHUB_TOKEN: {test_github_token[:5]}...")
    else:
        print("GITHUB_TOKEN was retrieved but is empty (might be a blank secret value).")
except Exception as e:
    print(f"GITHUB_TOKEN could NOT be retrieved by the notebook. Error: {e}")
    print("Please ensure 'GITHUB_TOKEN' exists in Colab Secrets and 'Notebook access' is ON.")


GITHUB_TOKEN could NOT be retrieved by the notebook. Error: Secret GITHUB_TOKEN does not exist.
Please ensure 'GITHUB_TOKEN' exists in Colab Secrets and 'Notebook access' is ON.


In [69]:
from pyngrok import ngrok
import threading
import time
import uvicorn
from google.colab import userdata
import os

# Get ngrok authtoken from Colab secrets or environment variable
NGROK_TOKEN = None
if 'NGROK_TOKEN' not in os.environ:
    try:
        NGROK_TOKEN = userdata.get('NGROK_TOKEN')
        print('ngrok token loaded from Colab secrets.')
    except Exception as e:
        print(f'Error loading NGROK_TOKEN from secrets: {e}')
else:
    NGROK_TOKEN = os.environ.get('NGROK_TOKEN')
    print('ngrok token already in environment.')

if NGROK_TOKEN:
    ngrok.set_auth_token(NGROK_TOKEN)
    print("ngrok authentication token set.")
else:
    print("Warning: NGROK_TOKEN not found. ngrok tunnel might not start. Please add NGROK_TOKEN to Colab secrets and ensure notebook access is ON, or set it as an environment variable.")


# Start the FastAPI server in a background thread
def start_server():
    # 'app' is defined in a previous cell (e4c21b3b)
    uvicorn.run(app, host="0.0.0.0", port=8000)

print("Starting FastAPI server in background...")
threading.Thread(target=start_server, daemon=True).start()
time.sleep(3) # Give the server a moment to start up

# Open the ngrok tunnel
public_url = ngrok.connect(8000).public_url
print(f"\n\n=== OPEN THIS URL IN YOUR META QUEST BROWSER ===\n{public_url}\n==================================================")
print("You can also access the API directly at:")
print(f"API Endpoint: {public_url}/api/nodes")
print(f"VR Frontend: {public_url}/")

Error loading NGROK_TOKEN from secrets: Secret NGROK_TOKEN does not exist.
Starting FastAPI server in background...


INFO:     Started server process [14149]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
ERROR:pyngrok.process.ngrok:t=2026-04-11T09:40:35+0000 lvl=eror msg="failed to reconnect session" obj=tunnels.session err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
ERROR:pyngrok.process.ngrok:t=2026-04-11T09:40:35+0000 lvl=eror msg="session closing" obj=tunnels.session err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-st

PyngrokNgrokError: The ngrok process errored on start: authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n.

In [66]:
from google.colab import userdata
import os

# Attempt to load the GITHUB_TOKEN from Colab secrets
# The user explicitly stated they toggled it ON.
if 'GITHUB_TOKEN' not in os.environ:
    try:
        os.environ['GITHUB_TOKEN'] = userdata.get('GITHUB_TOKEN')
        print('GitHub token loaded into environment from Colab secrets.')
    except Exception as e:
        print(f'Error loading GITHUB_TOKEN from secrets: {e}')
else:
    print('GitHub token already in environment.')

Error loading GITHUB_TOKEN from secrets: Secret GITHUB_TOKEN does not exist.


In [67]:
import requests
import os

def list_github_repositories(github_token=None, include_private=True, include_forks=True):
    """Fetches and lists GitHub repositories for the authenticated user."""
    # If no token is provided directly, try to get it from the environment
    if not github_token:
        github_token = os.environ.get('GITHUB_TOKEN')

    if not github_token:
        print("Error: A GitHub Personal Access Token is required to access private repositories.")
        print("Please provide it as an argument or set the GITHUB_TOKEN environment variable.")
        return []

    headers = {
        'Authorization': f'token {github_token}',
        'Accept': 'application/vnd.github.v3+json'
    }

    # Using the /user/repos endpoint fetches repositories for the authenticated user
    # affiliation=owner,collaborator,organization_member by default.
    # To ensure we get what we want, we can specify parameters.
    # 'visibility=all' gets both public and private if authenticated.
    params = {
        'visibility': 'all' if include_private else 'public',
        'per_page': 100, # Max per page
        'sort': 'updated'
    }

    repos = []
    page = 1
    print("Fetching repositories...")

    while True:
        params['page'] = page
        response = requests.get('https://api.github.com/user/repos', headers=headers, params=params)

        if response.status_code != 200:
            print(f"Failed to fetch repositories. Status Code: {response.status_code}")
            print(f"Response: {response.text}")
            break

        page_repos = response.json()
        if not page_repos:
            break # No more repositories

        for repo in page_repos:
            # Filter forks if requested
            if not include_forks and repo.get('fork', False):
                continue
            repos.append(repo)

        page += 1

    print(f"Found {len(repos)} repositories matching criteria.\n")

    # Display the extracted relevant information
    print(f"{'Repository Name':<40} | {'Visibility':<10} | {'Is Fork':<10}")
    print("-" * 66)
    for repo in repos:
        name = repo.get('full_name', 'Unknown')
        visibility = 'Private' if repo.get('private', False) else 'Public'
        is_fork = 'Yes' if repo.get('fork', False) else 'No'
        print(f"{name:<40} | {visibility:<10} | {is_fork:<10}")

    return repos

# Example usage (will fail gracefully if no token is set):
# To run this successfully, uncomment the line below and replace 'YOUR_TOKEN' with a real token
# list_github_repositories(github_token='YOUR_TOKEN')

# For now, it will attempt to use the environment variable
_ = list_github_repositories()

Error: A GitHub Personal Access Token is required to access private repositories.
Please provide it as an argument or set the GITHUB_TOKEN environment variable.


### Frontend Interface Development: ALDA MVP with Gradio

The Automated Legal Document Analysis (ALDA) MVP requires a user-friendly frontend to allow legal professionals to interact seamlessly with the underlying AI models. We will use **Gradio** to build this interface due to its rapid prototyping capabilities and intuitive design for machine learning applications.

#### UI Layout
The Gradio interface will be designed for simplicity and clarity, focusing on the core task of document analysis. The layout will include:

1.  **Secure File Upload Block:** A drag-and-drop file upload component allowing users to securely upload legal documents (PDF, DOCX, or TXT formats). This is the primary input point.
2.  **Analysis Controls (Optional):** A few optional settings, such as selecting the type of document (e.g., NDA, Employment Contract, Lease Agreement) to potentially refine the backend prompt, though the MVP will primarily focus on general contract analysis.
3.  **Dedicated Output Components:** The results will be displayed in clear, distinct sections:
    *   **Executive Summary:** A text box presenting a concise, plain-English summary of the document's main purpose and key terms.
    *   **Extracted Entities:** A table or structured list displaying key extracted information (e.g., Parties, Effective Date, Jurisdiction, Governing Law).
    *   **Risk Assessment:** A highlighted section showing the overall risk score (e.g., Low, Medium, High) and bullet points detailing specific risky clauses or anomalies identified by the model.

#### Interaction Flow
The interaction between the Gradio frontend and the FastAPI backend follows a straightforward request-response cycle:

1.  **User Input:** The user uploads a legal document via the Gradio file upload block and clicks the 'Analyze Document' button.
2.  **Trigger Backend Processing:** The Gradio application captures the uploaded file. It then sends an HTTP POST request to the designated FastAPI backend endpoint (e.g., `/analyze`), attaching the file payload.
3.  **Backend Execution:** The FastAPI backend receives the file, orchestrates the text extraction, and passes the text to the tuned DeepSeek-R1-Distill-Llama-8B model for analysis. It computes the risk scores and generates the structured JSON output.
4.  **Parsing and Presentation:** The backend returns the analysis report as a JSON response to the Gradio frontend. The Gradio application parses this JSON response and populates the respective output components (Summary, Entities, Risk Score) with the extracted data, presenting the final, readable report to the user.

## Cloud Deployment Roadmap

### Subtask:
Develop a concrete roadmap to transition the local codebase into a production-ready cloud service using Docker and cloud providers.


### ALDA SaaS MVP: Cloud Deployment Roadmap

Transitioning from a local Colab environment to a production-ready cloud service requires a structured approach. Here is the tactical roadmap to deploy the ALDA SaaS MVP:

#### 1. Containerization Strategy (Docker & Docker Compose)
To ensure consistency across environments and simplify deployment, the application will be fully containerized.
*   **Multi-Container Architecture:** We will use separate Docker containers for the backend and frontend.
    *   **Backend Container:** Packages the FastAPI application, ensuring all Python dependencies (like LangChain, vector DB clients, and ML libraries) are isolated and version-controlled.
    *   **Frontend Container:** Houses the Gradio web interface, configured to communicate seamlessly with the backend API.
*   **Orchestration:** `docker-compose` will be used to orchestrate these containers, managing networking and environment variables, allowing the entire stack to be spun up with a single command.

#### 2. Cloud Hosting & Infrastructure
Selecting the right infrastructure is critical, especially if the application relies on intensive AI models.
*   **Compute Providers:**
    *   **General Purpose (AWS / GCP / Azure):** Ideal for hosting the FastAPI backend, database, and frontend if external API calls (like OpenAI) are used heavily instead of local models.
    *   **GPU-Focused (RunPod / Lambda Labs / Vast.ai):** If the MVP eventually requires running custom, open-source LLMs locally, migrating to cost-effective GPU instances is essential for performance.
*   **Deployment Flow:** The Dockerized application will be deployed to a virtual machine (EC2, Compute Engine) or a managed container service (AWS ECS, Google Cloud Run) for scalable execution.

#### 3. Authentication & Monetization Setup
To transform the MVP into a viable business, secure user access and payment processing must be integrated.
*   **Authentication:** Integrate Identity and Access Management (IAM) solutions like **Auth0** or **Firebase Authentication**. This provides secure login, user management, and session handling without building it from scratch.
*   **Monetization Gateway:** Implement **Stripe** for billing and subscription management. By tying Auth0 user IDs to Stripe customer IDs, the backend can enforce access control based on active subscription tiers, unlocking premium ALDA features for paying users.

## Final Task

### Subtask:
Review the comprehensive SaaS MVP roadmap and summarize the architecture.


## Summary:

### Data Analysis Key Findings
* **Niche & MVP Strategy:** The SaaS MVP targets Automated Legal Document Analysis (ALDA) for small to medium-sized law firms, addressing the high costs and time associated with manual document review.
* **Backend Architecture:** A FastAPI backend was designed to handle secure file uploads and asynchronous processing via `BackgroundTasks`. It uses Pydantic models to ensure strict data validation across upload and reporting endpoints.
* **Frontend Design:** A Gradio-based interface was outlined to provide a simple user experience, featuring drag-and-drop file uploads and dedicated output sections for executive summaries, extracted key entities, and risk assessments.
* **Cloud Deployment Plan:** The roadmap specifies a multi-container Docker architecture orchestrated by `docker-compose`. Deployment targets include standard cloud providers (AWS/GCP) or GPU-specific hosts (RunPod), with Stripe for billing and Auth0 for user authentication.

### Insights or Next Steps
* **Insight:** Decoupling the heavy NLP extraction workload from the main API thread using background tasks ensures that the application remains highly responsive to user interactions on the Gradio frontend.
* **Next Step:** Initialize the local development environment by setting up the multi-container Docker architecture to test the request-response cycle between the Gradio UI and the FastAPI backend.


# Task
Enumerate my GitHub repositories, including forks and private ones. Then, analyze PR #28 and specific files in the BLOGAGENT repository for sensitive information. Also, review the task data (prompt, output, logs) from the task URL I provide. Finally, summarize all findings, ensuring compliance with any defined constraints.

## Enumerate Repositories

### Subtask:
List GitHub repositories for the user based on provided preferences.


**Reasoning**:
Adding a code block to enumerate GitHub repositories using the GitHub REST API, handling authentication, and fetching private/forked repositories based on user preferences.



In [ ]:
import requests
import os

def list_github_repositories(github_token=None, include_private=True, include_forks=True):
    """Fetches and lists GitHub repositories for the authenticated user."""
    # If no token is provided directly, try to get it from the environment
    if not github_token:
        github_token = os.environ.get('GITHUB_TOKEN')

    if not github_token:
        print("Error: A GitHub Personal Access Token is required to access private repositories.")
        print("Please provide it as an argument or set the GITHUB_TOKEN environment variable.")
        return []

    headers = {
        'Authorization': f'token {github_token}',
        'Accept': 'application/vnd.github.v3+json'
    }

    # Using the /user/repos endpoint fetches repositories for the authenticated user
    # affiliation=owner,collaborator,organization_member by default.
    # To ensure we get what we want, we can specify parameters.
    # 'visibility=all' gets both public and private if authenticated.
    params = {
        'visibility': 'all' if include_private else 'public',
        'per_page': 100, # Max per page
        'sort': 'updated'
    }

    repos = []
    page = 1
    print("Fetching repositories...")

    while True:
        params['page'] = page
        response = requests.get('https://api.github.com/user/repos', headers=headers, params=params)

        if response.status_code != 200:
            print(f"Failed to fetch repositories. Status Code: {response.status_code}")
            print(f"Response: {response.text}")
            break

        page_repos = response.json()
        if not page_repos:
            break # No more repositories

        for repo in page_repos:
            # Filter forks if requested
            if not include_forks and repo.get('fork', False):
                continue
            repos.append(repo)

        page += 1

    print(f"Found {len(repos)} repositories matching criteria.\n")

    # Display the extracted relevant information
    print(f"{'Repository Name':<40} | {'Visibility':<10} | {'Is Fork':<10}")
    print("-" * 66)
    for repo in repos:
        name = repo.get('full_name', 'Unknown')
        visibility = 'Private' if repo.get('private', False) else 'Public'
        is_fork = 'Yes' if repo.get('fork', False) else 'No'
        print(f"{name:<40} | {visibility:<10} | {is_fork:<10}")

    return repos

# Example usage (will fail gracefully if no token is set):
# To run this successfully, uncomment the line below and replace 'YOUR_TOKEN' with a real token
# list_github_repositories(github_token='YOUR_TOKEN')

# For now, it will attempt to use the environment variable
_ = list_github_repositories()

Error: A GitHub Personal Access Token is required to access private repositories.
Please provide it as an argument or set the GITHUB_TOKEN environment variable.


## Analyze Specific Repository (BLOGAGENT)

### Subtask:
Examine PR #28 and specific files in the BLOGAGENT repository for sensitive information.


**Reasoning**:
Adding a code block to fetch and analyze PR #28 from the BLOGAGENT repository for sensitive information using the GitHub API and regex-based secret scanning.



In [89]:
import requests
import os
import re

def analyze_pr_for_secrets(repo_name='BLOGAGENT', pr_number=28, owner='AxelJohnson1988', github_token=None):
    # Using the correct casing for the environment variable
    github_token = os.environ.get('Github_Token')

    if not github_token:
        print("Error: GitHub Personal Access Token is required.")
        return

    headers = {
        'Authorization': f'token {github_token}',
        'Accept': 'application/vnd.github.v3+json'
    }

    print(f"Analyzing PR #{pr_number} in {owner}/{repo_name}...")

    # Fetch PR files
    files_url = f'https://api.github.com/repos/{owner}/{repo_name}/pulls/{pr_number}/files'
    response = requests.get(files_url, headers=headers)

    if response.status_code != 200:
        print(f"Failed to fetch PR files. Status: {response.status_code}")
        print(f"Response: {response.text}")
        return

    files = response.json()
    print(f"Found {len(files)} modified files in PR #{pr_number}.")

    # Basic regex for sensitive info (passwords, tokens, api_keys)
    secret_pattern = re.compile(r'(?i)(api_key|apikey|token|password|secret)[\s\:\=]+["\\]?([a-zA-Z0-9\-\_]{8,})["\\]?')

    findings = False
    for file in files:
        filename = file.get('filename')
        patch = file.get('patch', '')

        if not patch:
            continue

        # Scan patch content
        matches = secret_pattern.findall(patch)
        if matches:
            findings = True
            print(f"\n[!] Potential sensitive data found in: {filename}")
            for match in matches:
                key_type, value = match
                # Mask value for display
                masked_value = value[:4] + '*' * (len(value) - 4) if len(value) > 4 else '****'
                print(f"    - Detected {key_type}: {masked_value}")

    if not findings:
        print("\n[+] No obvious sensitive information found in the PR patch contents.")

# Execute with the now-confirmed owner and repo_name
_ = analyze_pr_for_secrets(owner='AxelJohnson1988', repo_name='BLOGAGENT', pr_number=28)


Analyzing PR #28 in AxelJohnson1988/BLOGAGENT...
Failed to fetch PR files. Status: 404
Response: {"message":"Not Found","documentation_url":"https://docs.github.com/rest/pulls/pulls#list-pull-requests-files","status":"404"}


## Review Task Data

### Subtask:
Analyze prompt, output, and logs from the provided task URL if the user pastes the content.


### Task Data Review

No task URL or specific task data (prompt, output, logs) was provided in the initial request.

Please provide the prompt, output, and logs from the task URL you would like analyzed, and I will review them accordingly.

### Task Data Review

No task URL or specific task data (prompt, output, logs) was provided in the initial request.

Please provide the prompt, output, and logs from the task URL you would like analyzed, and I will review them accordingly.

### Task Data Review

No task URL or specific task data (prompt, output, logs) was provided in the initial request.

Please provide the prompt, output, and logs from the task URL you would like analyzed, and I will review them accordingly.

### Task Data Review

No task URL or specific task data (prompt, output, logs) was provided in the initial request.

Please provide the prompt, output, and logs from the task URL you would like analyzed, and I will review them accordingly.

### Task Data Review

No task URL or specific task data (prompt, output, logs) was provided in the initial request.

Please provide the prompt, output, and logs from the task URL you would like analyzed, and I will review them accordingly.

### Task Data Review

No task URL or specific task data (prompt, output, logs) was provided in the initial request.

Please provide the prompt, output, and logs from the task URL you would like analyzed, and I will review them accordingly.

## Final Task

### Subtask:
Summarize the findings from the repository analysis and task review, ensuring compliance with defined constraints.


## Summary:

### Q&A

**Were the GitHub repositories enumerated?**
No. While the logic to fetch repositories (including forks and private ones) via the GitHub REST API was developed, it could not be executed because a GitHub Personal Access Token (`GITHUB_TOKEN`) was not provided in the environment.

**Was sensitive information found in PR #28 of the BLOGAGENT repository?**
The scan could not be performed. A regex-based script was created to scan the pull request patch for secrets (like API keys and passwords), but it halted gracefully because it requires valid GitHub authentication to access the repository data.

**What were the findings from the task URL data?**
No analysis could be performed because the task URL and associated data (prompt, output, logs) were not provided in the initial request.

### Data Analysis Key Findings

* **Authentication Dependency:** Both the repository enumeration and the PR secret scanning tasks are heavily dependent on authenticated API requests. Without a `GITHUB_TOKEN`, fetching private repositories or specific PR files is blocked by GitHub.
* **Secret Scanning Logic:** The developed secret scanner relies on matching patch content against common keyword patterns (e.g., `api_key`, `token`, `password`) followed by strings of 8 or more characters.
* **Missing Inputs:** The task review phase was entirely stalled due to the absence of the target URL, prompt, output, and log data.

### Insights or Next Steps

* **Provide Authentication:** Set a valid `GITHUB_TOKEN` environment variable with the appropriate scopes (e.g., `repo`) to allow the scripts to fetch private repositories and pull request data.
* **Supply Missing Data:** Provide the specific task URL along with its prompt, output, and logs so the final review and summarization can be completed.


In [84]:
from google.colab import userdata
import os

# Attempt to load the Github_Token from Colab secrets
# The user explicitly stated they toggled it ON.
if 'Github_Token' not in os.environ:
    try:
        os.environ['Github_Token'] = userdata.get('Github_Token')
        print('GitHub token loaded into environment from Colab secrets.')
    except Exception as e:
        print(f'Error loading Github_Token from secrets: {e}')
else:
    print('GitHub token already in environment.')

GitHub token loaded into environment from Colab secrets.


In [85]:
import requests
import os

def list_github_repositories(github_token=None, include_private=True, include_forks=True):
    """Fetches and lists GitHub repositories for the authenticated user."""
    # If no token is provided directly, try to get it from the environment
    if not github_token:
        github_token = os.environ.get('Github_Token') # Corrected casing here

    if not github_token:
        print("Error: A GitHub Personal Access Token is required to access private repositories.")
        print("Please provide it as an argument or set the Github_Token environment variable.")
        return []

    headers = {
        'Authorization': f'token {github_token}',
        'Accept': 'application/vnd.github.v3+json'
    }

    # Using the /user/repos endpoint fetches repositories for the authenticated user
    # affiliation=owner,collaborator,organization_member by default.
    # To ensure we get what we want, we can specify parameters.
    # 'visibility=all' gets both public and private if authenticated.
    params = {
        'visibility': 'all' if include_private else 'public',
        'per_page': 100, # Max per page
        'sort': 'updated'
    }

    repos = []
    page = 1
    print("Fetching repositories...")

    while True:
        params['page'] = page
        response = requests.get('https://api.github.com/user/repos', headers=headers, params=params)

        if response.status_code != 200:
            print(f"Failed to fetch repositories. Status Code: {response.status_code}")
            print(f"Response: {response.text}")
            break

        page_repos = response.json()
        if not page_repos:
            break # No more repositories

        for repo in page_repos:
            # Filter forks if requested
            if not include_forks and repo.get('fork', False):
                continue
            repos.append(repo)

        page += 1

    print(f"Found {len(repos)} repositories matching criteria.\n")

    # Display the extracted relevant information
    print(f"{'Repository Name':<40} | {'Visibility':<10} | {'Is Fork':<10}")
    print("-" * 66)
    for repo in repos:
        name = repo.get('full_name', 'Unknown')
        visibility = 'Private' if repo.get('private', False) else 'Public'
        is_fork = 'Yes' if repo.get('fork', False) else 'No'
        print(f"{name:<40} | {visibility:<10} | {is_fork:<10}")

    return repos

# Example usage (will fail gracefully if no token is set):
# To run this successfully, uncomment the line below and replace 'YOUR_TOKEN' with a real token
# list_github_repositories(github_token='YOUR_TOKEN')

# For now, it will attempt to use the environment variable
_ = list_github_repositories()

Fetching repositories...
Found 8 repositories matching criteria.

Repository Name                          | Visibility | Is Fork   
------------------------------------------------------------------
AxelJohnson1988/FourSight2.0             | Public     | No        
AxelJohnson1988/laughing-adventure       | Public     | No        
AxelJohnson1988/stagehand                | Public     | Yes       
AxelJohnson1988/symmetrical-pancake      | Public     | No        
AxelJohnson1988/copilot-cli              | Public     | Yes       
AxelJohnson1988/warden-pipeline          | Public     | No        
AxelJohnson1988/firecrawl                | Public     | Yes       
AxelJohnson1988/contentapps              | Private    | No        


In [86]:
# Install Tesseract-OCR engine and Python wrapper
!sudo apt update
!sudo apt install tesseract-ocr
!pip install pytesseract Pillow

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [88.5 kB]
Get:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,497 kB]
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:7 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:11 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.0 MB]
Hit:13 https://ppa.launchpadcontent.net/ub

In [87]:
import pytesseract
from PIL import Image
import os

# List of image files provided by the user
image_files = [
    '/content/IMG_0206.png',
    '/content/IMG_0205.png'
]

def extract_text_from_image(image_path):
    if not os.path.exists(image_path):
        return f"Error: Image file not found at {image_path}"

    try:
        img = Image.open(image_path)
        text = pytesseract.image_to_string(img)
        return text
    except Exception as e:
        return f"Error processing {image_path}: {e}"

print("Ready to extract text from images.")

Ready to extract text from images.


In [88]:
for img_file in image_files:
    print(f"\n--- Extracting text from {img_file} ---")
    extracted_text = extract_text_from_image(img_file)
    print(extracted_text)


--- Extracting text from /content/IMG_0206.png ---
4:37AM Sat Apr 11

GMS

Dayforce Apple

USCU Mobile Banking Amazon.co...Smile more.

Raw Passio...n - Gay Porn

G © Phoenix_ & colab.research.google.com

=e

transparen...| - YouTube 20 Ways To...Indeed.com Base Smoo...er | TEMPTU

CO & Copy of Phoenix GPU_Cryptographic_Scanner.ipynb + @

PRO+

 

 

 

 

 

© Anotherc ©Agents-A © Yet anoth ‘ine-grair

Aaaaa111 pi...iendTV.com

 

=F WN © 2 195% GH)

OH +0

two yougn...kVideos.com

es 2, Share

File Edit View Insert Runtime Tools Help
Q Commands + Code ~ + Text Pp Runall 4 ay voa
o— ale} ;
— Secrets in Gemini 7 3 xX
M4 Error: A GitHub Personal Access Token is rec
A Configure your code by storing environment variables, file Please provide it as an argument or set the Be g) WS) PIES Yo
paths, or keys. Values stored here are private, visible only to requests. To give you the most
you and the notebooks that you select. accurate update, | need to
es @Frontend Interface Development: ALDA i

In [90]:
import requests
import os

def check_github_token_scopes():
    github_token = os.environ.get('Github_Token')

    if not github_token:
        print("Error: GitHub Personal Access Token is required to check scopes.")
        return

    headers = {
        'Authorization': f'token {github_token}',
        'Accept': 'application/vnd.github.v3+json'
    }

    # The /user endpoint can reveal token scopes in its response headers
    print("Fetching token scopes from GitHub API...")
    response = requests.get('https://api.github.com/user', headers=headers)

    if response.status_code == 200:
        # Scopes are usually in the 'X-OAuth-Scopes' header
        scopes = response.headers.get('X-OAuth-Scopes', 'No scopes found or token has default public access.')
        user_data = response.json()
        print(f"\nSuccessfully authenticated as GitHub user: {user_data.get('login')}")
        print(f"Token Scopes: {scopes}")
        print("\nCommon scopes for repository access are 'repo', 'public_repo', 'read:org'.")
        print("If 'repo' is present, it grants access to private repositories.")
    else:
        print(f"Failed to retrieve token information. Status Code: {response.status_code}")
        print(f"Response: {response.text}")

check_github_token_scopes()


Fetching token scopes from GitHub API...

Successfully authenticated as GitHub user: AxelJohnson1988
Token Scopes: No scopes found or token has default public access.

Common scopes for repository access are 'repo', 'public_repo', 'read:org'.
If 'repo' is present, it grants access to private repositories.
